# Pipeline de Análise de Produção Acadêmica (Lattes + ORCID + Scopus + Eventos)

Este notebook consolida em um único pipeline a produção bibliográfica de
**três fontes** por professor — Lattes (JSONs brutos), ORCID (API pública) e
Scopus (`pybliometrics`) — cruza com a base de percentil Scopus e com a base
de eventos classificados, detecta coautoria de alunos, e persiste tudo em
`pesquisadores_teste.duckdb`.

## O que mudou em relação à versão anterior (só Lattes)

- **Extração por fonte**: cada fonte (Lattes/ORCID/Scopus) agora produz seus
  próprios DataFrames de artigos de periódico e de trabalhos de congresso,
  **no mesmo schema final** (mesmas colunas de antes, mais uma coluna
  `fonte`). Isso é o que permite simplesmente concatenar as três fontes.
- **Unificação com deduplicação**: como o mesmo artigo pode aparecer em mais
  de uma fonte (ex.: Lattes e Scopus), a união usa `doi` normalizado (e,
  na ausência de DOI, `título normalizado + id_lattes`) como chave de
  deduplicação — sem isso, os índices de produtividade do `app.py` contariam
  o mesmo artigo mais de uma vez.
- **Tratamentos rodam uma vez, sobre a base já unificada e deduplicada**: o
  cruzamento com a planilha de percentil Scopus, o cruzamento com a base de
  eventos e a detecção de coautoria de alunos passam a considerar
  `COALESCE(titulo_revista_lattes, titulo_revista_scopus)` como nome do
  periódico (e o mesmo princípio para eventos), então artigos vindos de
  qualquer fonte são tratados da mesma forma.
- **Propagação de volta**: depois de calculado, o resultado do tratamento é
  mesclado de volta tanto na tabela unificada quanto nas três tabelas por
  fonte (pré-deduplicação) — cada uma preservando a granularidade "uma linha
  por publicação por fonte".
- **Banco de dados final com 11 tabelas** em vez de 5: as 5 tabelas de
  sempre (`tb_professores`, `tb_alunos`, `tb_orientacoes`,
  `tb_artigo_periodico`, `tb_artigo_conferencia` — mesmo formato de antes,
  para não quebrar `app.py`) mais 6 tabelas novas, uma por fonte e tipo de
  produção (`tb_artigo_periodico_lattes/orcid/scopus`,
  `tb_artigo_conferencia_lattes/orcid/scopus`).

## Estrutura deste notebook

1. Configuração e imports
2. Extração dos arquivos JSON brutos (Lattes) e consolidação em DataFrames
3. Tratamento de `df_pessoas` (informações pessoais)
4. Tratamento de `df_orientacoes`
5. Tratamento de `df_bib_artigos`/`df_bib_trab_congresso` e reshape para o schema final (fonte Lattes)
6. Extração ORCID (fonte ORCID, mesmo schema final)
7. Extração Scopus (fonte Scopus, mesmo schema final)
8. Unificação das três fontes com deduplicação
9. Cruzamento de periódicos com a base de percentil Scopus (sobre a base unificada)
10. Cruzamento de trabalhos de congresso com a base de eventos classificados (sobre a base unificada)
11. Detecção de coautoria de alunos nas produções
12. Propagação do tratamento de volta para as tabelas por fonte
13. Persistência consolidada no banco DuckDB (11 tabelas)


## 1. Configuração e Imports

Todas as bibliotecas usadas em qualquer etapa do notebook são importadas uma
única vez aqui, incluindo as duas novas dependências de extração
(`orcid` e `pybliometrics`) e `python-dotenv` para ler credenciais do
arquivo `.env`.


In [1]:
import json
import re
import time
import unicodedata
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
import orcid
import pybliometrics
from pybliometrics.scopus import ScopusSearch
from rapidfuzz import fuzz
from dotenv import load_dotenv

load_dotenv()

# Caminho onde estão os currículos Lattes (em JSON) dos professores.
# Cada arquivo é o "dump" bruto de um currículo já raspado/processado.
CAMINHO_JSONS_PROFESSORES = 'dados_brutos/professores/ufrj/*.json'

# Caminho da pasta com os currículos Lattes (em JSON) dos alunos, usada
# na Seção 11 para detectar coautoria entre professores e alunos.
CAMINHO_PASTA_ALUNOS = 'dados_brutos/alunos'

# Base de defesas do PESC (título Mestrado/Doutorado e data da defesa), usada
# na Seção 11 para enriquecer a tabela de alunos com 'titulo' e 'data_defesa'.
CAMINHO_CSV_DEFESAS = 'dados_brutos/defesas/defesas_pesc_agg.csv'

# Planilha de percentis Scopus por periódico (usada na Seção 9).
ARQUIVO_PERCENTIL_SCOPUS = 'periodicos_percentil.xlsx'

# Base de eventos/conferências já classificados por estrato (usada na Seção 10).
ARQUIVO_EVENTOS_CLASSIFICADOS = 'eventos_classificados_dois_idiomas.csv'

# Lista de pessoas com id_lattes/orcid_id/scopus_author_id (usada nas Seções 6 e 7).
# Gerado a partir de 'ID_lattes - Página1.csv'; orcid_id/scopus_author_id
# começam vazios e precisam ser preenchidos manualmente.
ARQUIVO_LISTA_PESSOAS = 'lista_pessoas.csv'

# Planilha administrativa oficial do programa: aluno, nível (M/D), ano de
# ingresso e orientador (usada na Seção 14).
CAMINHO_XLSX_ALUNOS_PESC = 'dados_brutos/lista_alunos_pesc.xlsx'

# Alias manual (nome_no_xlsx,id_lattes) para nomes de orientador que não
# batem automaticamente contra tb_professores (usado na Seção 14). Opcional.
CAMINHO_ALIAS_ORIENTADORES = 'dados_brutos/alias_orientadores.csv'

# Alias manual (nome_no_lattes_professor,id_lattes_aluno) para nomes de
# orientando digitados pelo professor no proprio Lattes que nao batem por
# nenhuma variacao/permutacao conhecida do nome do aluno -- typo ou aluno que
# mudou de nome depois que a orientacao foi registrada (usado na Secao 11.2.3).
# Opcional; a chave de resolucao continua sendo sempre o id_lattes do aluno.
CAMINHO_ALIAS_ORIENTANDOS = 'dados_brutos/alias_orientandos.csv'

# Anos de credenciamento de cada docente (usado na Seção 15): CSV com
# id_lattes, nome_referencia e a lista de anos separada por ';'. Enquanto não
# houver o registro administrativo oficial, é gerado com valores ALEATÓRIOS por
# 'gerar_credenciamento_aleatorio.py'. Opcional: sem ele a Seção 15 só avisa e
# deixa 'tb_credenciamento_anos' vazia.
CAMINHO_CSV_CREDENCIAMENTO = 'dados_brutos/credenciamento_professores.csv'

# Arquivo DuckDB de destino final (usado na Seção 13).
# Mantido como um arquivo de teste por padrão -- troque para
# 'pesquisadores.duckdb' quando quiser promover o resultado para produção.
ARQUIVO_DUCKDB_DESTINO = 'pesquisadores_teste.duckdb'

print("OK: configuração e imports carregados.")

OK: configuração e imports carregados.


## 2. Extração dos JSONs Brutos (Lattes) e Consolidação em DataFrames

Cada arquivo JSON representa o currículo Lattes de **um professor**, já
estruturado em blocos (`informacoes_pessoais`, `bancas`, `eventos`,
`orientacoes`, `premios_titulos`, `projetos_pesquisa`, `producao_bibliografica`,
`producao_tecnica`, `patentes_registros`).

A estratégia é:

1. Percorrer todos os arquivos JSON da pasta de professores.
2. Para cada bloco de interesse, "achatar" (flatten) a estrutura em uma lista
   de dicionários e marcar cada registro com o `id_lattes` do professor de origem.
3. Acumular essas listas e, ao final, concatenar tudo em um único DataFrame
   por tipo de produção (ex.: todos os artigos de periódico de todos os
   professores em um só `df_bib_artigos`).

Esta seção é idêntica à versão anterior deste notebook — a extração do Lattes
continua sendo a base de tudo, agora é só uma das três fontes que alimentam o
pipeline final.


In [2]:
print("Localizando arquivos JSON de professores...")
caminhos_arquivos = glob.glob(CAMINHO_JSONS_PROFESSORES)

if not caminhos_arquivos:
    raise FileNotFoundError(
        f"Nenhum arquivo JSON encontrado em '{CAMINHO_JSONS_PROFESSORES}'. "
        "Verifique se o caminho está correto antes de continuar."
    )

print(f"{len(caminhos_arquivos)} arquivo(s) encontrado(s). Iniciando a extração...")

Localizando arquivos JSON de professores...
31 arquivo(s) encontrado(s). Iniciando a extração...


In [3]:
# ---------------------------------------------------------
# 2.1 Listas de acumulação — uma por tipo de produção/registro
# ---------------------------------------------------------
# Dados Gerais
lista_pessoas = []
lista_bancas = []
lista_eventos = []
lista_orientacoes = []
lista_premios = []
lista_projetos = []

# Produção Bibliográfica
lista_bib_artigos = []
lista_bib_livros = []
lista_bib_capitulos = []
lista_bib_trabalhos_congresso = []
lista_bib_resumos_expandidos = []
lista_bib_resumos_congresso = []
lista_bib_artigos_aceitos = []
lista_bib_apresentacoes = []
lista_bib_textos_jornais = []
lista_bib_outras = []

# Produção Técnica
lista_tec_softwares_patente = []
lista_tec_softwares_sem_patente = []
lista_tec_produtos = []
lista_tec_processos = []
lista_tec_trabalhos = []
lista_tec_outras = []
lista_tec_entrevistas = []

# Patentes e Registros
lista_pat_patentes = []
lista_pat_programas = []
lista_pat_desenhos = []

In [4]:
# ---------------------------------------------------------
# 2.2 Extração e achatamento (flatten) de cada arquivo JSON
# ---------------------------------------------------------
for arquivo in caminhos_arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        dados = json.load(f)

        id_lattes = dados.get('informacoes_pessoais', {}).get('id_lattes')
        if not id_lattes:
            # Sem id_lattes não há como vincular nenhum registro a um professor;
            # o arquivo é descartado.
            continue

        # --- INFORMAÇÕES PESSOAIS ---
        df_pessoa = pd.json_normalize(dados['informacoes_pessoais'])
        lista_pessoas.append(df_pessoa)

        # --- BANCAS (estrutura: {categoria: [itens]}) ---
        if 'bancas' in dados:
            for categoria, itens in dados['bancas'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_banca'] = categoria
                    if 'membros_banca' in df_temp.columns:
                        # Lista de membros é convertida para string para caber em uma célula tabular
                        df_temp['membros_banca'] = df_temp['membros_banca'].astype(str)
                    lista_bancas.append(df_temp)

        # --- EVENTOS PARTICIPADOS (estrutura: {categoria: [itens]}) ---
        if 'eventos' in dados:
            for categoria, itens in dados['eventos'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_evento'] = categoria
                    lista_eventos.append(df_temp)

        # --- ORIENTAÇÕES (estrutura: {status: {nivel: [itens]}}) ---
        if 'orientacoes' in dados:
            for status, dicionario_niveis in dados['orientacoes'].items():
                for nivel, itens in dicionario_niveis.items():
                    if itens:
                        df_temp = pd.DataFrame(itens)
                        df_temp['id_lattes'] = id_lattes
                        df_temp['status'] = status
                        df_temp['nivel'] = nivel
                        lista_orientacoes.append(df_temp)

        # --- PRÊMIOS E TÍTULOS (lista simples) ---
        if 'premios_titulos' in dados and dados['premios_titulos']:
            df_temp = pd.DataFrame(dados['premios_titulos'])
            df_temp['id_lattes'] = id_lattes
            lista_premios.append(df_temp)

        # --- PROJETOS DE PESQUISA (lista simples, com campos longos) ---
        if 'projetos_pesquisa' in dados and dados['projetos_pesquisa']:
            df_temp = pd.DataFrame(dados['projetos_pesquisa'])
            df_temp['id_lattes'] = id_lattes
            for col in ['descricao', 'integrantes', 'financiadores']:
                if col in df_temp.columns:
                    df_temp[col] = df_temp[col].astype(str)
            lista_projetos.append(df_temp)

        # --- PRODUÇÃO BIBLIOGRÁFICA (estrutura: {chave: [itens]}) ---
        prod_bib = dados.get('producao_bibliografica', {})

        def add_to_list(chave, lista_destino):
            """Extrai uma chave de produção bibliográfica e empilha na lista destino."""
            itens = prod_bib.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list('artigos_periodicos', lista_bib_artigos)
        add_to_list('livros_publicados', lista_bib_livros)
        add_to_list('capitulos_livros', lista_bib_capitulos)
        add_to_list('trabalhos_completos_congressos', lista_bib_trabalhos_congresso)
        add_to_list('resumos_expandidos', lista_bib_resumos_expandidos)
        add_to_list('resumos_congressos', lista_bib_resumos_congresso)
        add_to_list('artigos_aceitos', lista_bib_artigos_aceitos)
        add_to_list('apresentacoes_trabalhos', lista_bib_apresentacoes)
        add_to_list('textos_jornais', lista_bib_textos_jornais)
        add_to_list('outras_producoes', lista_bib_outras)

        # --- PRODUÇÃO TÉCNICA (estrutura: {chave: [itens]}) ---
        prod_tec = dados.get('producao_tecnica', {})

        def add_to_list_tec(chave, lista_destino):
            """Extrai uma chave de produção técnica e empilha na lista destino."""
            itens = prod_tec.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_tec('softwares_com_patente', lista_tec_softwares_patente)
        add_to_list_tec('softwares_sem_patente', lista_tec_softwares_sem_patente)
        add_to_list_tec('produtos_tecnologicos', lista_tec_produtos)
        add_to_list_tec('processos_tecnicas', lista_tec_processos)
        add_to_list_tec('trabalhos_tecnicos', lista_tec_trabalhos)
        add_to_list_tec('outras_producoes_tecnicas', lista_tec_outras)
        add_to_list_tec('entrevistas', lista_tec_entrevistas)

        # --- PATENTES E REGISTROS (estrutura: {chave: [itens]}) ---
        patentes = dados.get('patentes_registros', {})

        def add_to_list_pat(chave, lista_destino):
            """Extrai uma chave de patentes/registros e empilha na lista destino."""
            itens = patentes.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_pat('patentes', lista_pat_patentes)
        add_to_list_pat('programas_computador', lista_pat_programas)
        add_to_list_pat('desenhos_industriais', lista_pat_desenhos)

print("Extração e achatamento concluídos para todos os arquivos.")

Extração e achatamento concluídos para todos os arquivos.


In [5]:
# ---------------------------------------------------------
# 2.3 Consolidação: cada lista de DataFrames parciais (um por professor)
#     é concatenada em um único DataFrame final por tipo de produção.
# ---------------------------------------------------------
def consolidar(lista):
    """Concatena uma lista de DataFrames parciais; retorna DataFrame vazio se a lista estiver vazia."""
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()

# DataFrames Gerais
df_pessoas = consolidar(lista_pessoas)
df_bancas = consolidar(lista_bancas)
df_eventos = consolidar(lista_eventos)
df_orientacoes = consolidar(lista_orientacoes)
df_premios = consolidar(lista_premios)
df_projetos = consolidar(lista_projetos)

# DataFrames de Produção Bibliográfica
df_bib_artigos = consolidar(lista_bib_artigos)
df_bib_livros = consolidar(lista_bib_livros)
df_bib_capitulos = consolidar(lista_bib_capitulos)
df_bib_trab_congresso = consolidar(lista_bib_trabalhos_congresso)
df_bib_resumos_exp = consolidar(lista_bib_resumos_expandidos)
df_bib_resumos_cong = consolidar(lista_bib_resumos_congresso)
df_bib_art_aceitos = consolidar(lista_bib_artigos_aceitos)
df_bib_apresentacoes = consolidar(lista_bib_apresentacoes)
df_bib_textos_jornais = consolidar(lista_bib_textos_jornais)
df_bib_outras = consolidar(lista_bib_outras)

# DataFrames de Produção Técnica
df_tec_soft_patente = consolidar(lista_tec_softwares_patente)
df_tec_soft_sem_patente = consolidar(lista_tec_softwares_sem_patente)
df_tec_produtos = consolidar(lista_tec_produtos)
df_tec_processos = consolidar(lista_tec_processos)
df_tec_trabalhos = consolidar(lista_tec_trabalhos)
df_tec_outras = consolidar(lista_tec_outras)
df_tec_entrevistas = consolidar(lista_tec_entrevistas)

# DataFrames de Patentes e Registros
df_pat_patentes = consolidar(lista_pat_patentes)
df_pat_programas = consolidar(lista_pat_programas)
df_pat_desenhos = consolidar(lista_pat_desenhos)

print("DataFrames consolidados. Resumo de volumes:")
print(f"  Professores (df_pessoas):              {len(df_pessoas)}")
print(f"  Orientações (df_orientacoes):           {len(df_orientacoes)}")
print(f"  Artigos de periódico (df_bib_artigos):  {len(df_bib_artigos)}")
print(f"  Trabalhos de congresso (df_bib_trab_congresso): {len(df_bib_trab_congresso)}")

DataFrames consolidados. Resumo de volumes:
  Professores (df_pessoas):              31
  Orientações (df_orientacoes):           2409
  Artigos de periódico (df_bib_artigos):  1604
  Trabalhos de congresso (df_bib_trab_congresso): 3119


## 3. Tratamento de `df_pessoas` (Informações Pessoais)

Limpeza padrão de cadastro: strings vazias→nulo, datas, remoção de marcador
"*" no rótulo, tipagem da chave primária e do texto de resumo. Ao final,
mesclamos `orcid_id`/`scopus_author_id` de `lista_pessoas.csv` — esses dois
IDs só existem para consultar as APIs externas (Seções 6 e 7), mas também
são guardados em `tb_professores` para rastreabilidade.


In [6]:
# 3.1 Substitui strings vazias ou só com espaços por NaN (nulo real)
df_pessoas.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 3.2 Converte a data de atualização do CV ('15/10/2025') para datetime.
#     errors='coerce' faz datas inválidas virarem nulo em vez de quebrar o script.
if 'atualizacao_cv' in df_pessoas.columns:
    df_pessoas['atualizacao_cv'] = pd.to_datetime(
        df_pessoas['atualizacao_cv'],
        format='%d/%m/%Y',
        errors='coerce'
    )

# 3.3 Limpeza do campo 'rotulo': remove o asterisco e espaços, e transforma
#     o texto literal "Sem rótulo" em nulo verdadeiro.
if 'rotulo' in df_pessoas.columns:
    df_pessoas['rotulo'] = df_pessoas['rotulo'].str.replace('*', '', regex=False).str.strip()
    df_pessoas['rotulo'] = df_pessoas['rotulo'].replace('Sem rótulo', np.nan)

# 3.4 Garante que a chave primária (id_lattes) seja sempre string,
#     evitando inconsistências de tipo em merges/joins posteriores.
df_pessoas['id_lattes'] = df_pessoas['id_lattes'].astype(str)

# 3.5 Remove espaços/quebras de linha nas bordas do texto de resumo do CV.
if 'texto_resumo' in df_pessoas.columns:
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

# 3.6 Mescla orcid_id/scopus_author_id da lista de mapeamento de pessoas.
#     Sem essa lista o cadastro em tb_professores não quebra -- as duas
#     colunas simplesmente ficam nulas.
if Path(ARQUIVO_LISTA_PESSOAS).exists():
    df_lista_pessoas = pd.read_csv(ARQUIVO_LISTA_PESSOAS, dtype=str)
    df_lista_pessoas['id_lattes'] = df_lista_pessoas['id_lattes'].astype(str).str.strip()
    df_lista_pessoas['orcid_id'] = df_lista_pessoas['orcid_id'].str.strip().replace('', np.nan)
    df_lista_pessoas['scopus_author_id'] = df_lista_pessoas['scopus_author_id'].str.strip().replace('', np.nan)
    # data_ingresso: ano de ingresso do professor no programa (inteiro).
    df_lista_pessoas['data_ingresso'] = pd.to_numeric(
        df_lista_pessoas['data_ingresso'], errors='coerce'
    ).astype('Int64')

    df_pessoas = df_pessoas.merge(
        df_lista_pessoas[['id_lattes', 'orcid_id', 'scopus_author_id', 'data_ingresso']],
        on='id_lattes', how='left'
    )
else:
    print(f"AVISO: '{ARQUIVO_LISTA_PESSOAS}' não encontrado -- orcid_id/scopus_author_id ficarão nulos.")
    df_pessoas['orcid_id'] = pd.NA
    df_pessoas['scopus_author_id'] = pd.NA
    df_pessoas['data_ingresso'] = pd.NA

print("Tratamento de df_pessoas concluído.")
df_pessoas.info()

Tratamento de df_pessoas concluído.
<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_lattes              31 non-null     str           
 1   nome_completo          31 non-null     str           
 2   nome_citacoes          31 non-null     str           
 3   sexo                   31 non-null     str           
 4   rotulo                 0 non-null      str           
 5   periodo                0 non-null      str           
 6   bolsa_produtividade    17 non-null     str           
 7   endereco_profissional  31 non-null     str           
 8   atualizacao_cv         31 non-null     datetime64[us]
 9   url                    31 non-null     str           
 10  texto_resumo           31 non-null     str           
 11  orcid_id               31 non-null     str           
 12  scopus_author_id       30 non-null     st

## 4. Tratamento de `df_orientacoes`

**Atenção à ordem**: a coluna original `titulo` é renomeada para
`titulo_trabalho` *antes* da limpeza de texto em lote, porque a limpeza já
referencia o nome novo (`titulo_trabalho`).

> **Nota de atenção (herdada do notebook original):** a Seção 13 insere
> `df_orientacoes` no banco esperando uma coluna `ano_inicio`. Essa coluna
> nunca é criada nem renomeada em nenhuma etapa de tratamento — ela só existe
> se o JSON bruto de orientações já trouxer um campo chamado literalmente
> `ano_inicio`. Se o seu JSON de origem usa outro nome para o ano de início da
> orientação, adicione aqui um `rename` (no mesmo padrão da linha abaixo que
> renomeia `titulo` → `titulo_trabalho`) antes de chegar à Seção 13, ou a
> inserção no banco falhará com `KeyError`/coluna inexistente.


In [7]:
# 4.1 Renomeia 'titulo' para 'titulo_trabalho' (nome mais descritivo,
#     usado pela limpeza de texto na sequência e pelo schema do banco).
df_orientacoes = df_orientacoes.rename(columns={'titulo': 'titulo_trabalho'})

print("Tratamento da tabela de orientações...")

# 4.2 Limpeza de texto: remove espaços duplos/quebras de linha escondidas e
#     preenche vazios com um rótulo explícito em vez de deixá-los como string vazia.
colunas_texto = ['titulo_trabalho', 'orientando', 'tipo_trabalho', 'instituicao', 'curso']
for col in colunas_texto:
    df_orientacoes[col] = df_orientacoes[col].astype(str).str.strip()
    df_orientacoes[col] = df_orientacoes[col].replace(
        {'': 'Não informado', 'nan': 'Não informado', 'None': 'Não informado'}
    )

# 4.3 Conversão segura do ano de conclusão (float -> Int64, que aceita nulos).
df_orientacoes['ano_conclusao'] = df_orientacoes['ano_conclusao'].astype('Int64')

# 4.4 Padroniza os valores de 'nivel' para rótulos amigáveis (usados em gráficos).
mapeamento_nivel = {
    'mestrado': 'Mestrado',
    'doutorado': 'Doutorado',
    'tcc': 'TCC',
    'iniciacao_cientifica': 'Iniciação Científica',
    'pos_doutorado': 'Pós-Doutorado',
    'especializacao': 'Especialização',
    'outros': 'Outros'
}
df_orientacoes['nivel'] = df_orientacoes['nivel'].map(mapeamento_nivel).fillna(df_orientacoes['nivel'])

# 4.5 Padroniza os valores de 'status' para rótulos amigáveis.
mapeamento_status = {
    'concluidas': 'Concluída',
    'em_andamento': 'Em Andamento'
}
df_orientacoes['status'] = df_orientacoes['status'].map(mapeamento_status).fillna(df_orientacoes['status'])

print("Tratamento concluído. Amostra dos dados tratados:")
display(df_orientacoes[['orientando', 'nivel', 'status', 'ano_conclusao']].head())

Tratamento da tabela de orientações...
Tratamento concluído. Amostra dos dados tratados:


,orientando,nivel,status,ano_conclusao
0,Rodrigo Fernandes Souto,Doutorado,Em Andamento,<NA>
1,Bruno Bandeira Monteiro,Doutorado,Em Andamento,<NA>
2,Rafael Paladini Meirelles,Mestrado,Em Andamento,<NA>
3,Eduardo Naslausky,Mestrado,Em Andamento,<NA>
4,Caio de Campos,TCC,Em Andamento,<NA>


## 5. Tratamento de `df_bib_artigos`/`df_bib_trab_congresso` e Reshape (fonte Lattes)

Antes de qualquer cruzamento com bases externas, os dois DataFrames de
produção bibliográfica do Lattes recebem uma limpeza básica: nulos reais,
padronização de maiúsculas no nome do periódico/evento, tipagem de ano e
remoção de espaços ocultos em texto (idêntico à versão anterior deste
notebook).

Em seguida, os dois DataFrames são reorganizados (*reshape*) para o **schema
final** — as mesmas colunas usadas por `tb_artigo_periodico` e
`tb_artigo_conferencia` — com os campos que dependem de cruzamento (percentil,
estrato, `match_adequado`, `coautoria_aluno` etc.) começando nulos. Essa é a
mesma estrutura que as extrações de ORCID e Scopus (Seções 6 e 7) também vão
produzir, o que torna a união das três fontes (Seção 8) um simples `concat`.


In [8]:
print("Aplicando tratamentos na tabela 'df_bib_artigos'...")

if not df_bib_artigos.empty:

    # 1. Strings vazias/só espaços -> nulo real
    df_bib_artigos.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # 2. Nome da revista em maiúsculas e sem espaços nas bordas
    #    (necessário para o cruzamento exato com a base Scopus na Seção 9)
    if 'revista' in df_bib_artigos.columns:
        df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()

    # 3. Ano como inteiro com suporte a nulo (Int64); valores inválidos -> nulo
    if 'ano' in df_bib_artigos.columns:
        df_bib_artigos['ano'] = pd.to_numeric(df_bib_artigos['ano'], errors='coerce').astype('Int64')

    # 4. Remove espaços ocultos nas colunas de texto livre
    colunas_texto = ['titulo', 'doi', 'issn', 'volume', 'numero', 'paginas']
    for col in colunas_texto:
        if col in df_bib_artigos.columns:
            df_bib_artigos[col] = df_bib_artigos[col].str.strip()

    # 5. Garante tipagem string na chave primária
    df_bib_artigos['id_lattes'] = df_bib_artigos['id_lattes'].astype(str)

print("Tratamento de 'df_bib_artigos' concluído.")
display(df_bib_artigos[['ano', 'revista', 'doi', 'issn']].head())

Aplicando tratamentos na tabela 'df_bib_artigos'...
Tratamento de 'df_bib_artigos' concluído.


,ano,revista,doi,issn
0,2021,HISTORY AND PHILOSOPHY OF LOGIC,http://dx.doi.org/10.1080/01445340.2021.1971005,1464-5149
1,2021,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2020.09.007,0166-218X
2,2020,DISCRETE MATHEMATICS,http://dx.doi.org/10.1016/j.disc.2019.111717,0012-365X
3,2020,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2019.03.022,0166-218X
4,2019,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,http://dx.doi.org/10.1016/j.entcs.2019.08.027,1571-0661


In [9]:
print("Aplicando tratamentos na tabela 'df_bib_trab_congresso'...")

if not df_bib_trab_congresso.empty:

    # 1. Strings vazias/só espaços -> nulo real
    df_bib_trab_congresso.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # 2. Nome do evento em maiúsculas e sem espaços nas bordas
    #    (necessário para o cruzamento com a base de eventos na Seção 10)
    if 'evento' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['evento'] = df_bib_trab_congresso['evento'].str.upper().str.strip()

    # 3. Ano como inteiro com suporte a nulo (Int64)
    if 'ano' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['ano'] = pd.to_numeric(df_bib_trab_congresso['ano'], errors='coerce').astype('Int64')

    # 4. Remove espaços ocultos nas colunas de texto livre
    colunas_texto = ['titulo', 'doi', 'isbn', 'paginas']
    for col in colunas_texto:
        if col in df_bib_trab_congresso.columns:
            df_bib_trab_congresso[col] = df_bib_trab_congresso[col].str.strip()

    # 5. Padronização da coluna 'autores': separador único (vírgula),
    #    sem espaços duplicados, e em maiúsculas (facilita buscas futuras,
    #    inclusive a detecção de coautoria de alunos na Seção 11).
    if 'autores' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.strip()
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(';', ',', regex=False)
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(r'\s+', ' ', regex=True)
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.upper()

    # 6. Garante tipagem string na chave primária
    df_bib_trab_congresso['id_lattes'] = df_bib_trab_congresso['id_lattes'].astype(str)

print("Tratamento de 'df_bib_trab_congresso' concluído.")
display(df_bib_trab_congresso[['ano', 'evento', 'autores']].head())

Aplicando tratamentos na tabela 'df_bib_trab_congresso'...


Tratamento de 'df_bib_trab_congresso' concluído.


,ano,evento,autores
0,2022,WORKSHOP BRASILEIRO DE LÓGICA,"CERIOLI, MÁRCIA R., FREITAS, RENATA DE , VIANA..."
1,2021,DIAGRAMS,"CERIOLI, M. R., SUGUITANI, L. , VIANA, PETRUCIO"
2,2017,LATIN AND AMERICAN ALGORITHMS,"CERIOLI, M. R., FERNANDES, C. G. , GOMES, R. ,..."
3,2017,CNMAC 2016 XXXVI CONGRESSO NACIONAL DE MATEMÁT...,"CERIOLI, MA'RCIA, NOBREGA, HUGO , SILVEIRA, GU..."
4,2015,XXXV CNMAC CONGRESSO NACIONAL DE MATEMÁTICA AP...,"BARROS, GABRIEL F. , POSNER, DANIEL F. D. , CE..."


### 5.1 Reshape para o schema final

`COLUNAS_PERIODICO`/`COLUNAS_CONGRESSO` definem o schema final único —
usado pelas três fontes (Lattes, ORCID, Scopus) e, mais tarde, pela tabela
unificada. Os campos que ainda dependem de cruzamento ficam nulos aqui;
são preenchidos nas Seções 9-11 e propagados de volta na Seção 12.


In [10]:
COLUNAS_PERIODICO = [
    'id_lattes', 'titulo_artigo', 'titulo_revista_lattes', 'ano_pub', 'doi',
    'autores', 'match_adequado', 'coautoria_aluno', 'id_scopus',
    'titulo_revista_scopus', 'maior_percentil', 'codigo_area_maior_percentil',
    'area_maior_percentil', 'issn', 'computation_area', 'citacoes_scopus',
    'fonte',
]

COLUNAS_CONGRESSO = [
    'id_lattes', 'titulo_artigo', 'ano', 'doi', 'autores',
    'titulo_evento_lattes', 'paginas', 'sigla_evento_google',
    'titulo_evento_google', 'estrato', 'tipo_match', 'coautoria_aluno',
    'citacoes_scopus', 'fonte',
]


def montar_df_vazio(colunas):
    """Cria um DataFrame vazio já com as colunas do schema final, evitando
    KeyError mais adiante quando uma fonte não retorna nenhum registro."""
    return pd.DataFrame(columns=colunas)


print("Reorganizando 'df_bib_artigos' para o schema final (fonte LATTES)...")
if not df_bib_artigos.empty:
    df_artigos_periodico_lattes = pd.DataFrame({
        'id_lattes': df_bib_artigos['id_lattes'],
        'titulo_artigo': df_bib_artigos.get('titulo'),
        'titulo_revista_lattes': df_bib_artigos.get('revista'),
        'ano_pub': df_bib_artigos.get('ano'),
        'doi': df_bib_artigos.get('doi'),
        'autores': df_bib_artigos.get('autores'),
        'match_adequado': pd.NA,
        'coautoria_aluno': pd.NA,
        'id_scopus': pd.NA,
        'titulo_revista_scopus': pd.NA,
        'maior_percentil': pd.NA,
        'codigo_area_maior_percentil': pd.NA,
        'area_maior_percentil': pd.NA,
        'issn': df_bib_artigos.get('issn'),
        'computation_area': pd.NA,
        'citacoes_scopus': pd.NA,
        'fonte': 'LATTES',
    })[COLUNAS_PERIODICO]
else:
    df_artigos_periodico_lattes = montar_df_vazio(COLUNAS_PERIODICO)

print("Reorganizando 'df_bib_trab_congresso' para o schema final (fonte LATTES)...")
if not df_bib_trab_congresso.empty:
    df_artigos_congresso_lattes = pd.DataFrame({
        'id_lattes': df_bib_trab_congresso['id_lattes'],
        'titulo_artigo': df_bib_trab_congresso.get('titulo'),
        'ano': df_bib_trab_congresso.get('ano'),
        'doi': df_bib_trab_congresso.get('doi'),
        'autores': df_bib_trab_congresso.get('autores'),
        'titulo_evento_lattes': df_bib_trab_congresso.get('evento'),
        'paginas': df_bib_trab_congresso.get('paginas'),
        'sigla_evento_google': pd.NA,
        'titulo_evento_google': pd.NA,
        'estrato': pd.NA,
        'tipo_match': pd.NA,
        'coautoria_aluno': pd.NA,
        'citacoes_scopus': pd.NA,
        'fonte': 'LATTES',
    })[COLUNAS_CONGRESSO]
else:
    df_artigos_congresso_lattes = montar_df_vazio(COLUNAS_CONGRESSO)

print(f"df_artigos_periodico_lattes: {len(df_artigos_periodico_lattes)} linhas")
print(f"df_artigos_congresso_lattes: {len(df_artigos_congresso_lattes)} linhas")
display(df_artigos_periodico_lattes.head())

Reorganizando 'df_bib_artigos' para o schema final (fonte LATTES)...
Reorganizando 'df_bib_trab_congresso' para o schema final (fonte LATTES)...
df_artigos_periodico_lattes: 1604 linhas
df_artigos_congresso_lattes: 3119 linhas


,id_lattes,titulo_artigo,titulo_revista_lattes,ano_pub,doi,autores,match_adequado,coautoria_aluno,id_scopus,titulo_revista_scopus,maior_percentil,codigo_area_maior_percentil,area_maior_percentil,issn,computation_area,fonte
0,0211300683784278,On the (In)Dependence of the Peano Axioms for ...,HISTORY AND PHILOSOPHY OF LOGIC,2021,http://dx.doi.org/10.1080/01445340.2021.1971005,"CERIOLI, MÁRCIA R.; NOBREGA, HUGO ; SILVEIRA, ...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1464-5149,<NA>,LATTES
1,0211300683784278,Short proofs on the structure of general parti...,DISCRETE APPLIED MATHEMATICS,2021,http://dx.doi.org/10.1016/j.dam.2020.09.007,"CERIOLI, MÁRCIA R.; MARTINS, TAÍSA",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0166-218X,<NA>,LATTES
2,0211300683784278,Transversals of longest paths,DISCRETE MATHEMATICS,2020,http://dx.doi.org/10.1016/j.disc.2019.111717,"CERIOLI, MÁRCIA R.; FERNANDES, CRISTINA G. ; G...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0012-365X,<NA>,LATTES
3,0211300683784278,Intersection of longest paths in graph classes,DISCRETE APPLIED MATHEMATICS,2020,http://dx.doi.org/10.1016/j.dam.2019.03.022,"CERIOLI, MÁRCIA R.; LIMA, PALOMA T.",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0166-218X,<NA>,LATTES
4,0211300683784278,On Edge-magic Labelings of Forests,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,2019,http://dx.doi.org/10.1016/j.entcs.2019.08.027,"CERIOLI, M. R.; FERNANDES, C. G. ; LEE, O. ; L...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1571-0661,<NA>,LATTES


## 6. Extração ORCID (fonte ORCID)

Adaptado de `orcid_1.ipynb`. Para cada pessoa em `lista_pessoas.csv` que já
tenha um `orcid_id` preenchido, lemos o registro público completo do ORCID e
extraímos a lista de `works` (trabalhos), classificando cada um em
**periódico** ou **congresso/evento** a partir do campo `type` retornado pela
API. O resultado já sai no schema final (`COLUNAS_PERIODICO`/
`COLUNAS_CONGRESSO`), com `fonte='ORCID'` e os campos de cruzamento nulos.

> **Limitação conhecida:** a API pública do ORCID não retorna o nome dos
> autores no resumo de cada trabalho (`work-summary`) na maioria dos casos —
> só o `credit-name` de quem cadastrou a publicação, quando preenchido. Por
> isso `autores` tende a ficar nulo para a maior parte das linhas de origem
> ORCID (mesma limitação já observada em `orcid_1.ipynb`; corrigir isso
> exigiria 1 requisição extra por trabalho e está fora do escopo desta
> mudança).


In [11]:
CLIENT_ID = __import__('os').getenv('ORCID_CLIENT_ID')
CLIENT_SECRET = __import__('os').getenv('ORCID_CLIENT_SECRET')

df_artigos_periodico_orcid = montar_df_vazio(COLUNAS_PERIODICO)
df_artigos_congresso_orcid = montar_df_vazio(COLUNAS_CONGRESSO)

if not CLIENT_ID or not CLIENT_SECRET:
    print("AVISO: ORCID_CLIENT_ID/ORCID_CLIENT_SECRET não configurados no .env -- "
          "extração ORCID pulada (df_artigos_periodico_orcid/df_artigos_congresso_orcid ficam vazios).")
else:
    api_orcid = orcid.PublicAPI(CLIENT_ID, CLIENT_SECRET, sandbox=False)
    token_orcid = api_orcid.get_search_token_from_orcid()
    print("Autenticação ORCID realizada com sucesso!")

Autenticação ORCID realizada com sucesso!


In [12]:
# Tipos de trabalho do ORCID que consideramos "artigo de periódico"
# (https://info.orcid.org/documentation/integration-guide/orcid-work-types/)
TIPOS_PERIODICO_ORCID = {
    'journal_article',
}

# Tipos de trabalho do ORCID que consideramos "trabalho de congresso/conferência"
TIPOS_CONGRESSO_ORCID = {
    'conference_paper',
}


def extrair_autores_orcid(trabalho):
    """Extrai os nomes de autores/contribuidores de um work-summary do ORCID,
    juntando-os em uma única string separada por vírgula."""
    contribuidores = (trabalho.get('contributors') or {}).get('contributor', [])
    nomes = []
    for contrib in contribuidores:
        credit_name = (contrib.get('credit-name') or {})
        nome = credit_name.get('value') if credit_name else None
        if nome:
            nomes.append(nome.strip())
    return ', '.join(nomes) if nomes else pd.NA


def extrair_doi_orcid(trabalho):
    ext_ids_container = trabalho.get('external-ids') or {}
    ext_ids = ext_ids_container.get('external-id', []) if ext_ids_container else []
    for ident in ext_ids:
        if ident.get('external-id-type') == 'doi':
            return ident.get('external-id-value')
    return pd.NA


if CLIENT_ID and CLIENT_SECRET:
    df_lista_pessoas_orcid = pd.read_csv(ARQUIVO_LISTA_PESSOAS, dtype=str) if Path(ARQUIVO_LISTA_PESSOAS).exists() else pd.DataFrame(columns=['id_lattes', 'orcid_id'])
    df_lista_pessoas_orcid = df_lista_pessoas_orcid[df_lista_pessoas_orcid['orcid_id'].notna() & (df_lista_pessoas_orcid['orcid_id'].str.strip() != '')]

    lista_artigos_periodico_orcid = []
    lista_artigos_congresso_orcid = []

    print(f"Total de pessoas com orcid_id preenchido: {len(df_lista_pessoas_orcid)}")

    for _, pessoa in df_lista_pessoas_orcid.iterrows():
        id_lattes = str(pessoa['id_lattes']).strip()
        orcid_id = str(pessoa['orcid_id']).strip()

        print(f"Processando ORCID {orcid_id} (id_lattes={id_lattes})...")

        try:
            perfil = api_orcid.read_record_public(orcid_id, 'record', token_orcid)
        except Exception as exc:
            print(f"  -> Falha ao ler perfil de {orcid_id}: {exc}")
            continue

        atividades = perfil.get('activities-summary', {})
        grupos_trabalhos = atividades.get('works', {}).get('group', [])

        for grupo in grupos_trabalhos:
            trabalho = grupo.get('work-summary', [{}])[0]

            titulo = trabalho.get('title', {}).get('title', {}).get('value', pd.NA)
            tipo = (trabalho.get('type') or '').lower()

            pub_date = trabalho.get('publication-date') or {}
            ano_raw = pub_date.get('year', {}).get('value') if pub_date else None
            ano = pd.NA if not ano_raw else int(ano_raw)

            doi = extrair_doi_orcid(trabalho)
            autores = extrair_autores_orcid(trabalho)

            if tipo in TIPOS_CONGRESSO_ORCID:
                evento_dict = trabalho.get('journal-title') or {}
                lista_artigos_congresso_orcid.append({
                    'id_lattes': id_lattes, 'titulo_artigo': titulo, 'ano': ano, 'doi': doi,
                    'autores': autores, 'titulo_evento_lattes': evento_dict.get('value', pd.NA),
                    'paginas': pd.NA, 'sigla_evento_google': pd.NA, 'titulo_evento_google': pd.NA,
                    'estrato': pd.NA, 'tipo_match': pd.NA, 'coautoria_aluno': pd.NA,
                    'citacoes_scopus': pd.NA, 'fonte': 'ORCID',
                })
            elif tipo in TIPOS_PERIODICO_ORCID:
                revista_dict = trabalho.get('journal-title') or {}
                lista_artigos_periodico_orcid.append({
                    'id_lattes': id_lattes, 'titulo_artigo': titulo,
                    'titulo_revista_lattes': revista_dict.get('value', pd.NA), 'ano_pub': ano,
                    'doi': doi, 'autores': autores, 'match_adequado': pd.NA, 'coautoria_aluno': pd.NA,
                    'id_scopus': pd.NA, 'titulo_revista_scopus': pd.NA, 'maior_percentil': pd.NA,
                    'codigo_area_maior_percentil': pd.NA, 'area_maior_percentil': pd.NA,
                    'issn': pd.NA, 'computation_area': pd.NA,
                    'citacoes_scopus': pd.NA, 'fonte': 'ORCID',
                })

        time.sleep(0.2)  # respeita o rate-limit público da API do ORCID

    if lista_artigos_periodico_orcid:
        df_artigos_periodico_orcid = pd.DataFrame(lista_artigos_periodico_orcid)[COLUNAS_PERIODICO]
    if lista_artigos_congresso_orcid:
        df_artigos_congresso_orcid = pd.DataFrame(lista_artigos_congresso_orcid)[COLUNAS_CONGRESSO]

    print(f"\nArtigos de periódico extraídos do ORCID: {len(df_artigos_periodico_orcid)}")
    print(f"Trabalhos de congresso extraídos do ORCID: {len(df_artigos_congresso_orcid)}")

Total de pessoas com orcid_id preenchido: 31
Processando ORCID 0000-0002-2941-2522 (id_lattes=8386103364234098)...


Processando ORCID 0000-0002-6393-0876 (id_lattes=3957046121364560)...


Processando ORCID 0000-0001-9712-1930 (id_lattes=5727472788265998)...


Processando ORCID 0000-0002-7538-7305 (id_lattes=2002515486942024)...


Processando ORCID 0000-0002-9712-3140 (id_lattes=0211300683784278)...


...

Processando ORCID 0000-0001-9341-6619 (id_lattes=3621433615334969)...


Processando ORCID 0000-0002-5660-6488 (id_lattes=8588117212005149)...


Processando ORCID 0000-0001-6411-9252 (id_lattes=3553487130003978)...


Processando ORCID 0000-0003-2987-4732 (id_lattes=1511298432327033)...



Artigos de periódico extraídos do ORCID: 821
Trabalhos de congresso extraídos do ORCID: 672


## 7. Extração Scopus (fonte Scopus)

Adaptado de `scopus_2.ipynb` (`pybliometrics`). Para cada pessoa em
`lista_pessoas.csv` com `scopus_author_id` preenchido, buscamos todas as
publicações indexadas via `ScopusSearch('AU-ID(...)')` e classificamos cada
uma em periódico ou congresso a partir de `subtype` — `ar` (Article) vira
periódico, `cp` (Conference Paper) vira congresso e todo o resto (`le`
Letter, `ed` Editorial, `no` Note, `er` Erratum, `re` Review, ...) é
descartado. `aggregationType` **não** serve para isso: ele diz só em que
veículo a publicação saiu, marcando "Journal" tanto para um artigo quanto
para uma carta ao editor publicada na mesma revista.

Além dos dois DataFrames no schema final, a extração monta
`df_catalogo_scopus` com **todas** as publicações retornadas pela API,
inclusive as descartadas e com seus `subtype`/`subtypeDescription`. É esse
catálogo que a Seção 7.1 usa como gabarito para corrigir a classificação
vinda do ORCID.

> **Nota de nomenclatura:** diferente da versão original de `scopus_2.ipynb`,
> aqui `id_scopus` **não** é preenchido com o `eid` (ID do artigo) retornado
> pela busca — em todo o resto do pipeline `id_scopus` sempre significou
> "Scopus Source ID" (ID do periódico, vindo da planilha de percentil na
> Seção 9). Preencher `id_scopus` com o `eid` aqui misturaria dois
> identificadores diferentes na mesma coluna; ele fica nulo até o
> cruzamento da Seção 9 preenchê-lo com o significado correto.

A extração também guarda `citedby_count` na coluna `citacoes_scopus` do schema
final. São as citações **daquela publicação**, não do periódico — a API de busca
não devolve métrica de veículo (CiteScore/SJR vêm do endpoint `SerialTitle`,
consultado por `source_id`/`issn`). A coluna existe nas seis tabelas por fonte
para que as três continuem com o mesmo schema, mas só as linhas com
`fonte = 'SCOPUS'` têm valor: nem o Lattes nem o ORCID expõem contagem de
citações. Nada no `app.py` lê essa coluna por enquanto — ela fica registrada no
banco à espera de uso.

In [13]:
SCOPUS_API_KEY = __import__('os').getenv('SCOPUS_API_KEY')
SCOPUS_INST_TOKEN = __import__('os').getenv('SCOPUS_INST_TOKEN')

df_artigos_periodico_scopus = montar_df_vazio(COLUNAS_PERIODICO)
df_artigos_congresso_scopus = montar_df_vazio(COLUNAS_CONGRESSO)

if not SCOPUS_API_KEY:
    print("AVISO: SCOPUS_API_KEY não configurada no .env -- "
          "extração Scopus pulada (df_artigos_periodico_scopus/df_artigos_congresso_scopus ficam vazios).")
else:
    pybliometrics.init(keys=[SCOPUS_API_KEY], inst_tokens=[SCOPUS_INST_TOKEN] if SCOPUS_INST_TOKEN else None)
    print("Cliente Scopus (pybliometrics) inicializado com sucesso!")

Cliente Scopus (pybliometrics) inicializado com sucesso!


In [14]:
def extrair_ano_scopus(cover_date):
    """Extrai o ano (int) de uma string de data tipo '2018-08-01'."""
    if not cover_date or pd.isna(cover_date):
        return pd.NA
    try:
        return int(str(cover_date)[:4])
    except (ValueError, TypeError):
        return pd.NA


# A classificação usa `subtype` (e não `aggregationType`): o agregador diz
# apenas em que *veículo* a publicação saiu ("Journal", "Conference
# Proceeding"), então uma carta ao editor publicada numa revista aparece como
# "Journal" igual a um artigo. O `subtype` diz o que a publicação *é*:
#   ar = Article        cp = Conference Paper   le = Letter
#   re = Review         ed = Editorial          no = Note
#   sh = Short Survey   ch = Chapter            er = Erratum   ...
# Só interessam ao pipeline artigos de periódico e trabalhos de congresso.
SUBTIPOS_PERIODICO_SCOPUS = {'ar'}
SUBTIPOS_CONGRESSO_SCOPUS = {'cp'}

# Colunas do catálogo Scopus — o registro de TUDO que a Scopus retornou,
# inclusive o que foi descartado. É ele que, na Seção 7.1, permite descobrir
# que um "journal_article" vindo do ORCID é na verdade uma carta.
COLUNAS_CATALOGO_SCOPUS = [
    'id_lattes', 'eid', 'titulo', 'doi', 'subtype', 'subtype_descricao',
    'aggregation_type', 'publicacao', 'aceito',
]

df_catalogo_scopus = montar_df_vazio(COLUNAS_CATALOGO_SCOPUS)

if SCOPUS_API_KEY:
    df_lista_pessoas_scopus = pd.read_csv(ARQUIVO_LISTA_PESSOAS, dtype=str) if Path(ARQUIVO_LISTA_PESSOAS).exists() else pd.DataFrame(columns=['id_lattes', 'scopus_author_id'])
    df_lista_pessoas_scopus = df_lista_pessoas_scopus[df_lista_pessoas_scopus['scopus_author_id'].notna() & (df_lista_pessoas_scopus['scopus_author_id'].str.strip() != '')]

    lista_artigos_periodico_scopus = []
    lista_artigos_congresso_scopus = []
    lista_catalogo_scopus = []

    print(f"Total de pessoas com scopus_author_id preenchido: {len(df_lista_pessoas_scopus)}")

    for _, pessoa in df_lista_pessoas_scopus.iterrows():
        id_lattes = str(pessoa['id_lattes']).strip()
        scopus_author_id = str(pessoa['scopus_author_id']).strip()

        print(f"Processando Scopus Author ID {scopus_author_id} (id_lattes={id_lattes})...")

        try:
            s = ScopusSearch(f'AU-ID({scopus_author_id})')
        except Exception as exc:
            print(f"  -> Falha ao buscar publicações de {scopus_author_id}: {exc}")
            continue

        publicacoes = s.results or []
        if not publicacoes:
            print("  -> Nenhuma publicação encontrada.")
            continue

        for pub in publicacoes:
            titulo = pub.title or pd.NA
            revista = pub.publicationName or pd.NA
            ano = extrair_ano_scopus(pub.coverDate)
            doi = pub.doi or pd.NA
            issn = pub.issn or pd.NA
            autores = pub.author_names or pd.NA
            subtipo = (getattr(pub, 'subtype', None) or '').strip().lower()
            subtipo_descricao = getattr(pub, 'subtypeDescription', None) or pd.NA
            tipo_agregacao = (pub.aggregationType or '').strip().lower()
            # `citedby_count` vem em toda publicacao retornada pela API e conta as
            # citacoes *daquele artigo* -- nao e' metrica do periodico. E' a unica
            # coluna do schema que so a fonte Scopus consegue preencher; Lattes e
            # ORCID nao expoem nada equivalente, entao ficam nulas.
            citacoes = getattr(pub, 'citedby_count', None)
            citacoes = pd.NA if citacoes is None else int(citacoes)

            aceito = subtipo in SUBTIPOS_PERIODICO_SCOPUS or subtipo in SUBTIPOS_CONGRESSO_SCOPUS

            # O catálogo registra TODA publicação retornada, aceita ou não.
            lista_catalogo_scopus.append({
                'id_lattes': id_lattes, 'eid': getattr(pub, 'eid', None) or pd.NA,
                'titulo': titulo, 'doi': doi, 'subtype': subtipo or pd.NA,
                'subtype_descricao': subtipo_descricao, 'aggregation_type': tipo_agregacao or pd.NA,
                'publicacao': revista, 'aceito': aceito,
            })

            if subtipo in SUBTIPOS_CONGRESSO_SCOPUS:
                lista_artigos_congresso_scopus.append({
                    'id_lattes': id_lattes, 'titulo_artigo': titulo, 'ano': ano, 'doi': doi,
                    'autores': autores, 'titulo_evento_lattes': revista, 'paginas': pub.pageRange or pd.NA,
                    'sigla_evento_google': pd.NA, 'titulo_evento_google': pd.NA, 'estrato': pd.NA,
                    'tipo_match': pd.NA, 'coautoria_aluno': pd.NA,
                    'citacoes_scopus': citacoes, 'fonte': 'SCOPUS',
                })
            elif subtipo in SUBTIPOS_PERIODICO_SCOPUS:
                lista_artigos_periodico_scopus.append({
                    'id_lattes': id_lattes, 'titulo_artigo': titulo, 'titulo_revista_lattes': pd.NA,
                    'ano_pub': ano, 'doi': doi, 'autores': autores, 'match_adequado': pd.NA,
                    'coautoria_aluno': pd.NA, 'id_scopus': pd.NA, 'titulo_revista_scopus': revista,
                    'maior_percentil': pd.NA, 'codigo_area_maior_percentil': pd.NA,
                    'area_maior_percentil': pd.NA, 'issn': issn, 'computation_area': pd.NA,
                    'citacoes_scopus': citacoes, 'fonte': 'SCOPUS',
                })

        time.sleep(0.2)

    if lista_artigos_periodico_scopus:
        df_artigos_periodico_scopus = pd.DataFrame(lista_artigos_periodico_scopus)[COLUNAS_PERIODICO]
    if lista_artigos_congresso_scopus:
        df_artigos_congresso_scopus = pd.DataFrame(lista_artigos_congresso_scopus)[COLUNAS_CONGRESSO]
    if lista_catalogo_scopus:
        df_catalogo_scopus = pd.DataFrame(lista_catalogo_scopus)[COLUNAS_CATALOGO_SCOPUS]

    print(f"\nArtigos de periódico extraídos da Scopus: {len(df_artigos_periodico_scopus)}")
    print(f"Trabalhos de congresso extraídos da Scopus: {len(df_artigos_congresso_scopus)}")
    print(f"Catálogo Scopus (tudo que a API retornou): {len(df_catalogo_scopus)} publicações")

    if not df_catalogo_scopus.empty:
        print("\nDistribuição por subtipo (o que foi aceito e o que foi descartado):")
        display(
            df_catalogo_scopus.groupby(['subtype', 'subtype_descricao', 'aceito'], dropna=False)
            .size().rename('publicacoes').reset_index()
            .sort_values('publicacoes', ascending=False)
        )

Total de pessoas com scopus_author_id preenchido: 30
Processando Scopus Author ID 56405183700 (id_lattes=8386103364234098)...


Processando Scopus Author ID 7004168896 (id_lattes=3957046121364560)...


Processando Scopus Author ID 23991001200 (id_lattes=5727472788265998)...


Processando Scopus Author ID 6603938094 (id_lattes=2002515486942024)...


Processando Scopus Author ID 6603427338 (id_lattes=0211300683784278)...


...

Processando Scopus Author ID 7006343277 (id_lattes=8588117212005149)...


Processando Scopus Author ID 35778596500 (id_lattes=3553487130003978)...


Processando Scopus Author ID 23398339800 (id_lattes=1511298432327033)...



Artigos de periódico extraídos da Scopus: 1223
Trabalhos de congresso extraídos da Scopus: 1468
Catálogo Scopus (tudo que a API retornou): 2900 publicações

Distribuição por subtipo (o que foi aceito e o que foi descartado):


,subtype,subtype_descricao,aceito,publicacoes
3,cp,Conference Paper,True,1468
0,ar,Article,True,1223
2,ch,Book Chapter,False,83
5,ed,Editorial,False,73
9,re,Review,False,34
7,le,Letter,False,7
1,bk,Book,False,6
6,er,Erratum,False,4
4,dp,Data Paper,False,1
8,no,Note,False,1


## 7.1 Revisão da Classificação do ORCID pelo Catálogo Scopus

A API do ORCID descreve cada trabalho por um `type` autodeclarado por quem
cadastrou a publicação, e esse campo não separa bem os gêneros: cartas ao
editor, editoriais, notas e errata publicados em revista costumam chegar como
`journal_article` e entravam no pipeline como se fossem artigos. A Scopus, ao
contrário, traz o campo `subtype`/`subtypeDescription`, que diz o que a
publicação **é** (`ar` = Article, `cp` = Conference Paper, `le` = Letter,
`ed` = Editorial, `re` = Review, `no` = Note, `er` = Erratum, ...) —
diferente de `aggregationType`, que diz apenas em que veículo ela saiu e
portanto marca "Journal" tanto para um artigo quanto para uma carta.

Como praticamente tudo que está na Scopus também está no ORCID, o catálogo
Scopus (`df_catalogo_scopus`, montado na Seção 7 com **todas** as publicações
retornadas, inclusive as descartadas) serve de gabarito. Esta seção precisa
rodar depois das duas extrações — primeiro ORCID (Seção 6), depois Scopus
(Seção 7) — e para cada linha vinda do ORCID procura a publicação
correspondente no catálogo, primeiro por **DOI normalizado** e, se não
achar, por **título normalizado dentro do mesmo professor** (mesmas funções
de normalização usadas na deduplicação da Seção 8, ambas comparações exatas).

O desfecho depende do `subtype` encontrado:

| `subtype` no catálogo | Ação sobre a linha do ORCID |
| --- | --- |
| `ar` (Article) | fica em periódicos (movida para lá, se estava em congressos) |
| `cp` (Conference Paper) | fica em congressos (movida para lá, se estava em periódicos) |
| qualquer outro (`le`, `ed`, `no`, `er`, `re`, ...) | **descartada** — não é artigo nem trabalho de congresso |
| não encontrada no catálogo | mantida como veio (a Scopus não conhece a publicação, não há o que conferir) |

O relatório `df_revisao_orcid` lista linha a linha o que foi corrigido, com o
subtipo que motivou cada decisão, para conferência. A revisão só mexe nas
linhas de origem ORCID: Lattes e Scopus seguem intactos.

In [15]:
# ---------------------------------------------------------------------------
# 7.1.1 Normalização de DOI e título
# ---------------------------------------------------------------------------
# Importadas de `dedup_publicacoes`, o módulo com a regra de deduplicação
# compartilhada com o pipeline da base de comparação. Ficam aqui porque esta é
# a primeira seção que precisa delas: "casar com o catálogo Scopus" (Seção 7.1)
# e "ser a mesma publicação" (Seção 8) têm de seguir exatamente o mesmo
# critério, e agora seguem por construção -- é o mesmo código.
from dedup_publicacoes import normalizar_doi, normalizar_titulo_dedup


# ---------------------------------------------------------------------------
# 7.1.2 Consulta ao catálogo Scopus
# ---------------------------------------------------------------------------
def indexar_catalogo_scopus(df_catalogo):
    """Monta dois índices de consulta a partir do catálogo: um por DOI
    normalizado (o DOI é único globalmente, então serve mesmo quando a
    publicação foi catalogada pelo coautor) e outro por título normalizado
    dentro do mesmo professor (título sozinho é ambíguo demais para casar
    entre professores diferentes)."""
    por_doi = {}
    por_titulo = {}
    for registro in df_catalogo.itertuples(index=False):
        subtipo = registro.subtype
        if pd.isna(subtipo) or not str(subtipo).strip():
            continue
        achado = (str(subtipo).strip().lower(), registro.subtype_descricao)

        doi = normalizar_doi(registro.doi)
        if not pd.isna(doi):
            por_doi.setdefault(doi, achado)

        titulo = normalizar_titulo_dedup(registro.titulo)
        if titulo:
            por_titulo.setdefault((str(registro.id_lattes), titulo), achado)
    return por_doi, por_titulo


def consultar_catalogo_scopus(linha, por_doi, por_titulo):
    """Procura no catálogo a publicação de uma linha do ORCID: primeiro pelo
    DOI, depois pelo título dentro do mesmo professor. Retorna
    (subtype, subtypeDescription) ou None se a Scopus não a conhece."""
    doi = normalizar_doi(linha.get('doi'))
    if not pd.isna(doi) and doi in por_doi:
        return por_doi[doi]

    titulo = normalizar_titulo_dedup(linha.get('titulo_artigo'))
    if titulo:
        return por_titulo.get((str(linha.get('id_lattes')), titulo))
    return None


# ---------------------------------------------------------------------------
# 7.1.3 Conversão entre os dois schemas (quando a linha muda de lado)
# ---------------------------------------------------------------------------
def converter_periodico_para_congresso(linha):
    """Reescreve uma linha do schema de periódico no schema de congresso —
    usada quando a Scopus revela que o que o ORCID chamou de artigo de
    periódico é na verdade um trabalho de congresso."""
    convertida = {coluna: pd.NA for coluna in COLUNAS_CONGRESSO}
    convertida.update({
        'id_lattes': linha['id_lattes'], 'titulo_artigo': linha['titulo_artigo'],
        'ano': linha['ano_pub'], 'doi': linha['doi'], 'autores': linha['autores'],
        'titulo_evento_lattes': linha['titulo_revista_lattes'],
        'citacoes_scopus': linha['citacoes_scopus'], 'fonte': linha['fonte'],
    })
    return convertida


def converter_congresso_para_periodico(linha):
    """Caminho inverso de `converter_periodico_para_congresso`."""
    convertida = {coluna: pd.NA for coluna in COLUNAS_PERIODICO}
    convertida.update({
        'id_lattes': linha['id_lattes'], 'titulo_artigo': linha['titulo_artigo'],
        'ano_pub': linha['ano'], 'doi': linha['doi'], 'autores': linha['autores'],
        'titulo_revista_lattes': linha['titulo_evento_lattes'],
        'citacoes_scopus': linha['citacoes_scopus'], 'fonte': linha['fonte'],
    })
    return convertida


# ---------------------------------------------------------------------------
# 7.1.4 A revisão propriamente dita
# ---------------------------------------------------------------------------
COLUNAS_REVISAO_ORCID = [
    'id_lattes', 'titulo_artigo', 'doi', 'classificacao_orcid',
    'subtype_scopus', 'subtype_descricao_scopus', 'acao',
]


def revisar_classificacao_orcid(df_periodico, df_congresso, df_catalogo):
    """Confere as linhas vindas do ORCID contra o catálogo Scopus e devolve
    (periódicos revisados, congressos revisados, relatório das mudanças).

    Linhas que a Scopus não conhece são mantidas exatamente como vieram — o
    catálogo só é usado para *corrigir* o que ele consegue afirmar, nunca
    para descartar por ausência de evidência."""
    if df_catalogo.empty:
        print("Catálogo Scopus vazio -- revisão pulada (nada com que conferir).")
        return df_periodico, df_congresso, montar_df_vazio(COLUNAS_REVISAO_ORCID)

    por_doi, por_titulo = indexar_catalogo_scopus(df_catalogo)

    periodicos_revisados = []
    congressos_revisados = []
    ocorrencias = []

    for origem, df_origem in [('PERIODICO', df_periodico), ('CONGRESSO', df_congresso)]:
        for linha in df_origem.to_dict('records'):
            achado = consultar_catalogo_scopus(linha, por_doi, por_titulo)

            if achado is None:
                destino = origem  # a Scopus não conhece: mantém como veio
                subtipo, subtipo_descricao = pd.NA, pd.NA
            else:
                subtipo, subtipo_descricao = achado
                if subtipo in SUBTIPOS_PERIODICO_SCOPUS:
                    destino = 'PERIODICO'
                elif subtipo in SUBTIPOS_CONGRESSO_SCOPUS:
                    destino = 'CONGRESSO'
                else:
                    destino = 'DESCARTADO'

            if destino == 'PERIODICO':
                periodicos_revisados.append(
                    linha if origem == 'PERIODICO' else converter_congresso_para_periodico(linha)
                )
            elif destino == 'CONGRESSO':
                congressos_revisados.append(
                    linha if origem == 'CONGRESSO' else converter_periodico_para_congresso(linha)
                )

            if destino != origem:
                ocorrencias.append({
                    'id_lattes': linha['id_lattes'], 'titulo_artigo': linha['titulo_artigo'],
                    'doi': linha['doi'], 'classificacao_orcid': origem,
                    'subtype_scopus': subtipo, 'subtype_descricao_scopus': subtipo_descricao,
                    'acao': 'DESCARTADO' if destino == 'DESCARTADO' else f'MOVIDO PARA {destino}',
                })

    df_periodico_revisado = (
        pd.DataFrame(periodicos_revisados)[COLUNAS_PERIODICO] if periodicos_revisados
        else montar_df_vazio(COLUNAS_PERIODICO)
    )
    df_congresso_revisado = (
        pd.DataFrame(congressos_revisados)[COLUNAS_CONGRESSO] if congressos_revisados
        else montar_df_vazio(COLUNAS_CONGRESSO)
    )
    df_relatorio = (
        pd.DataFrame(ocorrencias)[COLUNAS_REVISAO_ORCID] if ocorrencias
        else montar_df_vazio(COLUNAS_REVISAO_ORCID)
    )
    return df_periodico_revisado.reset_index(drop=True), df_congresso_revisado.reset_index(drop=True), df_relatorio


print("Conferindo a classificação do ORCID contra o catálogo Scopus...")
print(f"  Antes -- ORCID periódicos: {len(df_artigos_periodico_orcid)} | "
      f"ORCID congressos: {len(df_artigos_congresso_orcid)}")

df_artigos_periodico_orcid, df_artigos_congresso_orcid, df_revisao_orcid = revisar_classificacao_orcid(
    df_artigos_periodico_orcid, df_artigos_congresso_orcid, df_catalogo_scopus,
)

print(f"  Depois -- ORCID periódicos: {len(df_artigos_periodico_orcid)} | "
      f"ORCID congressos: {len(df_artigos_congresso_orcid)}")

if df_revisao_orcid.empty:
    print("Nenhuma correção necessária: tudo que a Scopus conhece já estava classificado corretamente.")
else:
    print(f"\n{len(df_revisao_orcid)} linha(s) do ORCID corrigida(s):")
    display(df_revisao_orcid['acao'].value_counts())
    print("\nDescartes por subtipo Scopus:")
    display(
        df_revisao_orcid[df_revisao_orcid['acao'] == 'DESCARTADO']
        .groupby(['subtype_scopus', 'subtype_descricao_scopus'], dropna=False)
        .size().rename('linhas').reset_index().sort_values('linhas', ascending=False)
    )
    display(df_revisao_orcid.head(20))

Conferindo a classificação do ORCID contra o catálogo Scopus...
  Antes -- ORCID periódicos: 821 | ORCID congressos: 672
  Depois -- ORCID periódicos: 692 | ORCID congressos: 758

139 linha(s) do ORCID corrigida(s):


acao
MOVIDO PARA CONGRESSO    92
DESCARTADO               43
MOVIDO PARA PERIODICO     4
Name: count, dtype: int64


Descartes por subtipo Scopus:


,subtype_scopus,subtype_descricao_scopus,linhas
5,re,Review,22
2,ed,Editorial,10
3,le,Letter,7
1,ch,Book Chapter,2
0,bk,Book,1
4,no,Note,1


,id_lattes,titulo_artigo,doi,classificacao_orcid,subtype_scopus,subtype_descricao_scopus,acao
0,3957046121364560,The cost of perfection for matchings in graphs,10.1016/j.dam.2014.12.006,PERIODICO,cp,Conference Paper,MOVIDO PARA CONGRESSO
1,3957046121364560,On the recognition of unit disk graphs and the...,10.1016/j.dam.2014.08.014,PERIODICO,cp,Conference Paper,MOVIDO PARA CONGRESSO
2,3957046121364560,The complexity of forbidden subgraph sandwich ...,10.1016/j.dam.2013.09.004,PERIODICO,cp,Conference Paper,MOVIDO PARA CONGRESSO
3,3957046121364560,Complexity of colouring problems restricted to...,10.1016/j.dam.2012.02.016,PERIODICO,cp,Conference Paper,MOVIDO PARA CONGRESSO
4,3957046121364560,The hunting of a snark with total chromatic nu...,10.1016/j.dam.2013.04.006,PERIODICO,cp,Conference Paper,MOVIDO PARA CONGRESSO
5,3957046121364560,Graph theory and algorithms: Fourth Latin-Amer...,10.1007/s13173-012-0068-4,PERIODICO,ed,Editorial,DESCARTADO
6,3957046121364560,A decomposition for total-coloring partial-gri...,10.1002/net.20424,PERIODICO,cp,Conference Paper,MOVIDO PARA CONGRESSO
7,3957046121364560,On the forbidden induced subgraph sandwich pro...,10.1016/j.dam.2010.11.010,PERIODICO,cp,Conference Paper,MOVIDO PARA CONGRESSO
8,3957046121364560,Total chromatic number of unichord-free graphs,10.1016/j.dam.2011.03.024,PERIODICO,cp,Conference Paper,MOVIDO PARA CONGRESSO
9,3957046121364560,Preface,10.1016/j.dam.2007.07.017,PERIODICO,ed,Editorial,DESCARTADO


## 7.2 Revisão da Classificação de ORCID e Scopus pelo Lattes

A Seção 7.1 corrige o `type` do ORCID usando o `subtype` do catálogo Scopus como
gabarito, mas isso deixa dois buracos: o próprio `subtype` da Scopus nunca é
conferido contra nada, e nada é conferido contra o Lattes -- que é o gabarito
mais confiável porque é preenchido manualmente pelo próprio professor.

Isso importa porque periódicos e congressos são famílias de tabelas totalmente
separadas até a deduplicação da Seção 8, que casa `chave_dedup` **dentro de
cada tabela**, nunca entre elas. Se o Lattes tem um artigo como periódico e o
ORCID/Scopus tem o mesmo artigo (mesmo DOI/título) classificado como congresso,
as duas linhas nunca se encontram: a linha do Lattes fica sozinha com
`fontes='LATTES'` e a linha do ORCID/Scopus fica sozinha na tabela errada, com
`fontes` sem `LATTES`. É esse segundo caso que a Streamlit app usa como
critério de "artigo faltando no Lattes" (`fontes NOT LIKE '%LATTES%'`) -- um
falso positivo, já que o artigo já está no Lattes, só que na tabela errada.

Esta seção roda depois da 7.1 (para revisar o ORCID já corrigido pelo catálogo
Scopus) e usa as duas tabelas Lattes (`df_artigos_periodico_lattes`/
`df_artigos_congresso_lattes`, já existentes desde a Seção 5) como índice de
consulta, pelas mesmas duas chaves de igualdade da Seção 7.1 (DOI normalizado
e título normalizado dentro do mesmo professor). Para cada linha de ORCID e de
Scopus:

| Classificação encontrada no Lattes | Ação sobre a linha |
| --- | --- |
| igual à atual, ou Lattes não conhece a publicação | mantida como veio |
| diferente da atual | movida para o lado que o Lattes diz |
| ambígua (o próprio Lattes tem a mesma publicação cadastrada nas duas tabelas) | mantida como veio, mas sinalizada para conferência manual |

Diferente da 7.1, não existe descarte aqui: o Lattes só tem dois baldes
(periódico e congresso) neste pipeline, então a única coisa a decidir é para
qual lado a linha deve ir. O relatório `df_revisao_classificacao_lattes` lista
linha a linha o que foi corrigido (ou sinalizado como ambíguo), para
conferência. A revisão só mexe em linhas de origem ORCID/Scopus -- o Lattes
segue intacto.

In [16]:
# ---------------------------------------------------------------------------
# 7.2.1 Índice de classificação a partir das duas tabelas Lattes
# ---------------------------------------------------------------------------
def indexar_classificacao_lattes(df_periodico_lattes, df_congresso_lattes):
    """Monta os mesmos dois índices de consulta da Seção 7.1.2 (por DOI
    normalizado e por título normalizado dentro do mesmo professor), desta
    vez a partir das duas tabelas Lattes, que são o gabarito mais confiável
    porque são curadas manualmente pelo próprio professor. Cada chave mapeia
    para 'PERIODICO' ou 'CONGRESSO' -- ou para o sentinela 'AMBIGUO' quando o
    próprio Lattes tem o mesmo DOI/título cadastrado nas duas tabelas (erro de
    cadastro), caso em que não há como decidir e a linha correspondente não
    deve ser reclassificada."""
    por_doi = {}
    por_titulo = {}

    def registrar(indice, chave, classificacao):
        atual = indice.get(chave)
        if atual is None:
            indice[chave] = classificacao
        elif atual != classificacao:
            indice[chave] = 'AMBIGUO'

    for classificacao, df in (('PERIODICO', df_periodico_lattes), ('CONGRESSO', df_congresso_lattes)):
        for registro in df.itertuples(index=False):
            doi = normalizar_doi(registro.doi)
            if not pd.isna(doi):
                registrar(por_doi, doi, classificacao)

            titulo = normalizar_titulo_dedup(registro.titulo_artigo)
            if titulo:
                registrar(por_titulo, (str(registro.id_lattes), titulo), classificacao)

    return por_doi, por_titulo


def consultar_classificacao_lattes(linha, por_doi, por_titulo):
    """Procura a classificação Lattes de uma linha ORCID/Scopus: primeiro
    pelo DOI, depois pelo título dentro do mesmo professor. Retorna
    'PERIODICO', 'CONGRESSO', 'AMBIGUO' ou None se o Lattes não conhece a
    publicação."""
    doi = normalizar_doi(linha.get('doi'))
    if not pd.isna(doi) and doi in por_doi:
        return por_doi[doi]

    titulo = normalizar_titulo_dedup(linha.get('titulo_artigo'))
    if titulo:
        return por_titulo.get((str(linha.get('id_lattes')), titulo))
    return None


# ---------------------------------------------------------------------------
# 7.2.2 A revisão propriamente dita
# ---------------------------------------------------------------------------
COLUNAS_REVISAO_LATTES = [
    'id_lattes', 'titulo_artigo', 'doi', 'fonte', 'classificacao_original',
    'classificacao_lattes', 'acao',
]


def revisar_classificacao_por_lattes(df_periodico, df_congresso, por_doi, por_titulo):
    """Confere as linhas de ORCID/Scopus contra o índice de classificação do
    Lattes e devolve (periódicos revisados, congressos revisados, relatório
    das mudanças).

    Linhas que o Lattes não conhece, ou cuja classificação já bate com a
    atual, são mantidas como vieram. Linhas ambíguas (o próprio Lattes tem o
    mesmo DOI/título nas duas tabelas) também são mantidas, mas aparecem no
    relatório para conferência manual -- o objetivo aqui é corrigir o que dá
    para afirmar com confiança, nunca arriscar um palpite."""
    periodicos_revisados = []
    congressos_revisados = []
    ocorrencias = []

    for origem, df_origem in [('PERIODICO', df_periodico), ('CONGRESSO', df_congresso)]:
        for linha in df_origem.to_dict('records'):
            classificacao = consultar_classificacao_lattes(linha, por_doi, por_titulo)
            destino = classificacao if classificacao in ('PERIODICO', 'CONGRESSO') else origem

            if destino == 'PERIODICO':
                periodicos_revisados.append(
                    linha if origem == 'PERIODICO' else converter_congresso_para_periodico(linha)
                )
            else:
                congressos_revisados.append(
                    linha if origem == 'CONGRESSO' else converter_periodico_para_congresso(linha)
                )

            if classificacao == 'AMBIGUO' or destino != origem:
                ocorrencias.append({
                    'id_lattes': linha['id_lattes'], 'titulo_artigo': linha['titulo_artigo'],
                    'doi': linha['doi'], 'fonte': linha['fonte'], 'classificacao_original': origem,
                    'classificacao_lattes': classificacao,
                    'acao': 'AMBIGUO' if classificacao == 'AMBIGUO' else f'MOVIDO PARA {destino}',
                })

    df_periodico_revisado = (
        pd.DataFrame(periodicos_revisados)[COLUNAS_PERIODICO] if periodicos_revisados
        else montar_df_vazio(COLUNAS_PERIODICO)
    )
    df_congresso_revisado = (
        pd.DataFrame(congressos_revisados)[COLUNAS_CONGRESSO] if congressos_revisados
        else montar_df_vazio(COLUNAS_CONGRESSO)
    )
    df_relatorio = (
        pd.DataFrame(ocorrencias)[COLUNAS_REVISAO_LATTES] if ocorrencias
        else montar_df_vazio(COLUNAS_REVISAO_LATTES)
    )
    return df_periodico_revisado.reset_index(drop=True), df_congresso_revisado.reset_index(drop=True), df_relatorio


# ---------------------------------------------------------------------------
# 7.2.3 Execução -- aplica a mesma revisão às quatro tabelas ORCID/Scopus
# ---------------------------------------------------------------------------
print("Conferindo a classificação de ORCID e Scopus contra o Lattes...")
print(f"  Antes -- ORCID periódicos: {len(df_artigos_periodico_orcid)} | ORCID congressos: {len(df_artigos_congresso_orcid)}")
print(f"  Antes -- Scopus periódicos: {len(df_artigos_periodico_scopus)} | Scopus congressos: {len(df_artigos_congresso_scopus)}")

_por_doi_lattes, _por_titulo_lattes = indexar_classificacao_lattes(df_artigos_periodico_lattes, df_artigos_congresso_lattes)

df_artigos_periodico_orcid, df_artigos_congresso_orcid, df_revisao_lattes_orcid = revisar_classificacao_por_lattes(
    df_artigos_periodico_orcid, df_artigos_congresso_orcid, _por_doi_lattes, _por_titulo_lattes,
)
df_artigos_periodico_scopus, df_artigos_congresso_scopus, df_revisao_lattes_scopus = revisar_classificacao_por_lattes(
    df_artigos_periodico_scopus, df_artigos_congresso_scopus, _por_doi_lattes, _por_titulo_lattes,
)

df_revisao_classificacao_lattes = pd.concat(
    [df_revisao_lattes_orcid, df_revisao_lattes_scopus], ignore_index=True,
)

print(f"  Depois -- ORCID periódicos: {len(df_artigos_periodico_orcid)} | ORCID congressos: {len(df_artigos_congresso_orcid)}")
print(f"  Depois -- Scopus periódicos: {len(df_artigos_periodico_scopus)} | Scopus congressos: {len(df_artigos_congresso_scopus)}")

if df_revisao_classificacao_lattes.empty:
    print("\nNenhuma correção necessária: tudo que o Lattes conhece já estava classificado corretamente.")
else:
    print(f"\n{len(df_revisao_classificacao_lattes)} linha(s) de ORCID/Scopus revisada(s) pelo Lattes:")
    display(df_revisao_classificacao_lattes.groupby(['fonte', 'acao']).size().rename('linhas').reset_index())
    display(df_revisao_classificacao_lattes.head(20))


Conferindo a classificação de ORCID e Scopus contra o Lattes...
  Antes -- ORCID periódicos: 692 | ORCID congressos: 758
  Antes -- Scopus periódicos: 1223 | Scopus congressos: 1468


  Depois -- ORCID periódicos: 710 | ORCID congressos: 740
  Depois -- Scopus periódicos: 1219 | Scopus congressos: 1472

340 linha(s) de ORCID/Scopus revisada(s) pelo Lattes:


,fonte,acao,linhas
0,ORCID,AMBIGUO,12
1,ORCID,MOVIDO PARA CONGRESSO,38
2,ORCID,MOVIDO PARA PERIODICO,56
3,SCOPUS,AMBIGUO,34
4,SCOPUS,MOVIDO PARA CONGRESSO,102
5,SCOPUS,MOVIDO PARA PERIODICO,98


,id_lattes,titulo_artigo,doi,fonte,classificacao_original,classificacao_lattes,acao
0,3957046121364560,Using SPQR-trees to speed up algorithms based ...,10.1016/j.endm.2015.07.029,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO
1,3957046121364560,The generalized split probe problem,10.1016/j.endm.2013.10.007,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO
2,3957046121364560,On coloring problems of snark families,10.1016/j.endm.2011.05.009,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO
3,3957046121364560,Complexity dichotomy on degree-constrained VLS...,10.1016/j.endm.2010.05.050,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO
4,3957046121364560,"Total chromatic number of {square,unichord}-fr...",10.1016/j.endm.2010.05.085,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO
5,3957046121364560,Skew partition sandwich problem is NP-complete,10.1016/j.endm.2009.11.003,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO
6,3957046121364560,2K<inf>2</inf> vertex-set partition into nonem...,10.1016/j.endm.2008.01.050,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO
7,3957046121364560,"On maximizing clique, clique-Helly and heredit...",10.1016/j.endm.2008.01.026,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO
8,3957046121364560,Sufficient conditions for a graph to be edge-c...,10.1016/j.endm.2008.01.013,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO
9,3957046121364560,The polynomial dichotomy for three nonempty pa...,10.1016/j.endm.2008.01.015,ORCID,PERIODICO,CONGRESSO,MOVIDO PARA CONGRESSO


## 8. Unificação das Três Fontes com Deduplicação

As três fontes produzem exatamente o mesmo schema
(`COLUNAS_PERIODICO`/`COLUNAS_CONGRESSO`) e são simplesmente concatenadas.
O resultado bruto (`df_periodicos_bruto`/`df_congressos_bruto`) **preserva
uma linha por publicação por fonte** — é essa granularidade que a Seção 12
propaga de volta para as 6 tabelas por fonte.

### A regra que orienta toda a seção

A deduplicação é feita **dentro da lista de cada professor, isoladamente**.
O agrupamento é por `id_lattes` e a chave gerada leva o `id_lattes` como
prefixo, de modo que duas linhas de professores diferentes **nunca** podem
cair na mesma chave. Consequência prática: se dois professores do quadro
coassinaram o mesmo artigo, os dois continuam com aquele artigo na sua
própria lista — a pergunta "quais artigos este professor tem?" continua
respondível. O que se elimina é apenas a repetição **dentro** da lista de um
mesmo professor.

### Como duas linhas do mesmo professor viram a mesma publicação

Só por comparação **exata**, sobre valores previamente tratados — não há
nenhum limiar de similaridade em nenhuma etapa:

- mesmo **DOI normalizado** (`normalizar_doi`: minúsculo, sem `doi:`, sem
  `https://doi.org/`, sem pontuação na borda); ou
- mesmo **título normalizado** (`normalizar_titulo_dedup`: maiúsculas, sem
  acentuação, sem pontuação, espaços colapsados).

As duas funções de normalização são as definidas na Seção 7.1 — o mesmo
critério de igualdade que casa uma linha do ORCID com o catálogo Scopus.

Os dois critérios são combinados por componentes conexos porque cada fonte
preenche campos diferentes: se o Lattes trouxe o título sem DOI e o Scopus
trouxe o mesmo título com DOI, é o título que liga as duas linhas; se uma
terceira linha do ORCID só tem o DOI, é o DOI que a liga à do Scopus. As
três acabam na mesma publicação. A chave final é `id_lattes|DOI:<doi>`
quando o grupo tem DOI, e `id_lattes|TIT:<titulo>` quando não tem.

### As duas reduções

1. `deduplicar_bruto_por_fonte` deixa uma linha por (publicação, fonte) na
   base bruta — garante que as 6 tabelas por fonte, usadas nos relatórios,
   também não tenham artigos repetidos.
2. `unificar_com_dedup` reduz à base final: uma linha por publicação de cada
   professor, com cada coluna preenchida pelo primeiro valor não nulo na
   ordem LATTES > SCOPUS > ORCID (Lattes é curado manualmente pelo professor;
   Scopus tem metadados mais limpos que o resumo público do ORCID). A coluna
   `fontes` registra todas as bases em que aquela publicação foi encontrada.

### Segunda passada

`auditar_duplicatas` percorre as listas de novo, sem alterar nada, checando
os cinco invariantes: nenhum professor com DOI repetido, nenhum professor com
título repetido, nenhuma chave repetida na base unificada, nenhuma linha
repetida dentro da mesma fonte na base bruta e nenhuma chave compartilhada
por professores diferentes (este último confirma que a coautoria entre
professores do quadro não fundiu ninguém).

In [17]:
# ---------------------------------------------------------------------------
# 8. Unificação das fontes com deduplicação
# ---------------------------------------------------------------------------
# Toda a regra vive em `dedup_publicacoes`, compartilhada com o pipeline da
# base de comparação (`analyse_organizado_comparação.ipynb`). As duas bases
# existem para serem comparadas entre si: se cada uma tivesse a sua cópia
# destas funções, uma correção aplicada em só um dos lados faria os números
# divergirem por causa do código, e não por causa dos dados.
#
# O que o módulo garante, em ordem:
#   `calcular_chave_dedup`       -- agrupa por DOI ou título, sempre dentro de
#                                   um mesmo professor, descartando antes os
#                                   DOIs que uma fonte usa em mais de uma
#                                   publicação daquele professor;
#   `sanear_doi_gravado`         -- apaga da coluna `doi` o que não identifica a
#                                   publicação e canoniza o que sobra, devolvendo
#                                   o relatório do que foi descartado;
#   `deduplicar_bruto_por_fonte` -- uma linha por (publicação, fonte);
#   `unificar_com_dedup`         -- uma linha por publicação, com `fontes`;
#   `auditar_duplicatas`         -- segunda passada conferindo o resultado.
from dedup_publicacoes import (
    auditar_duplicatas,
    calcular_chave_dedup,
    deduplicar_bruto_por_fonte,
    sanear_doi_gravado,
    unificar_com_dedup,
)


# ---------------------------------------------------------------------------
# 8.5 Execução
# ---------------------------------------------------------------------------
print("Concatenando as três fontes (uma linha por publicação por fonte)...")
df_periodicos_bruto = pd.concat(
    [df_artigos_periodico_lattes, df_artigos_periodico_orcid, df_artigos_periodico_scopus],
    ignore_index=True,
)
df_congressos_bruto = pd.concat(
    [df_artigos_congresso_lattes, df_artigos_congresso_orcid, df_artigos_congresso_scopus],
    ignore_index=True,
)

# So a fonte Scopus preenche `citacoes_scopus`; o concat com Lattes/ORCID deixa
# a coluna como object (int misturado com pd.NA), o que quebra o TRY_CAST na
# carga e atrapalha a inspecao aqui no notebook. Int64 e' o inteiro nullable.
for _df_bruto in (df_periodicos_bruto, df_congressos_bruto):
    _df_bruto['citacoes_scopus'] = pd.to_numeric(
        _df_bruto['citacoes_scopus'], errors='coerce').astype('Int64')

print("Calculando a chave de deduplicação (por professor, via DOI ou título)...")
df_periodicos_bruto['chave_dedup'] = calcular_chave_dedup(df_periodicos_bruto)
df_congressos_bruto['chave_dedup'] = calcular_chave_dedup(df_congressos_bruto)

print("Descartando DOIs que não identificam a publicação...")
df_periodicos_bruto, df_dois_descartados_periodicos = sanear_doi_gravado(
    df_periodicos_bruto, 'periódico', 'ano_pub')
df_congressos_bruto, df_dois_descartados_congressos = sanear_doi_gravado(
    df_congressos_bruto, 'congresso', 'ano')
df_dois_descartados = pd.concat(
    [df_dois_descartados_periodicos, df_dois_descartados_congressos], ignore_index=True)

print("Removendo repetições dentro de uma mesma fonte...")
df_periodicos_bruto = deduplicar_bruto_por_fonte(df_periodicos_bruto)
df_congressos_bruto = deduplicar_bruto_por_fonte(df_congressos_bruto)

print("Unificando as fontes (uma linha por publicação de cada professor)...")
df_periodicos_unificado = unificar_com_dedup(df_periodicos_bruto)
df_congressos_unificado = unificar_com_dedup(df_congressos_bruto)

if df_dois_descartados.empty:
    print("Nenhum DOI descartado: todo DOI preenchido identifica uma única publicação.")
else:
    print(f"\n{len(df_dois_descartados)} linha(s) tiveram o DOI descartado "
          f"(o valor original fica abaixo, para correção no currículo):")
    display(df_dois_descartados['motivo'].value_counts())
    display(df_dois_descartados.sort_values(['tipo', 'id_lattes', 'ano']))

print("\nSegunda passada — conferindo se sobrou alguma duplicata:")
auditar_duplicatas(df_periodicos_unificado, df_periodicos_bruto, 'periódicos')
auditar_duplicatas(df_congressos_unificado, df_congressos_bruto, 'congressos')

print(f"\nPeriódicos -- bruto (por fonte): {len(df_periodicos_bruto)} | unificado: {len(df_periodicos_unificado)}")
print(f"Congressos -- bruto (por fonte): {len(df_congressos_bruto)} | unificado: {len(df_congressos_unificado)}")
print("\nCobertura por combinação de fontes (periódicos):")
display(df_periodicos_unificado['fontes'].value_counts())
display(df_periodicos_unificado[['id_lattes', 'titulo_artigo', 'doi', 'fontes']].head())

Concatenando as três fontes (uma linha por publicação por fonte)...
Calculando a chave de deduplicação (por professor, via DOI ou título)...


Removendo repetições dentro de uma mesma fonte...


Unificando as fontes (uma linha por publicação de cada professor)...



Segunda passada — conferindo se sobrou alguma duplicata:
  OK (periódicos): nenhum professor tem artigo repetido, nem na base unificada nem nas bases por fonte; nenhuma chave cruzou professores.
  OK (congressos): nenhum professor tem artigo repetido, nem na base unificada nem nas bases por fonte; nenhuma chave cruzou professores.

Periódicos -- bruto (por fonte): 3467 | unificado: 1746
Congressos -- bruto (por fonte): 5270 | unificado: 3428

Cobertura por combinação de fontes (periódicos):


fontes
LATTES,ORCID,SCOPUS    554
LATTES,SCOPUS          524
LATTES                 454
SCOPUS                  86
LATTES,ORCID            57
ORCID                   39
ORCID,SCOPUS            32
Name: count, dtype: int64

,id_lattes,titulo_artigo,doi,fontes
0,0211300683784278,On the (In)Dependence of the Peano Axioms for ...,http://dx.doi.org/10.1080/01445340.2021.1971005,"LATTES,ORCID,SCOPUS"
1,0211300683784278,Short proofs on the structure of general parti...,http://dx.doi.org/10.1016/j.dam.2020.09.007,"LATTES,SCOPUS"
2,0211300683784278,Transversals of longest paths,http://dx.doi.org/10.1016/j.disc.2019.111717,"LATTES,SCOPUS"
3,0211300683784278,Intersection of longest paths in graph classes,http://dx.doi.org/10.1016/j.dam.2019.03.022,"LATTES,SCOPUS"
4,0211300683784278,On Edge-magic Labelings of Forests,http://dx.doi.org/10.1016/j.entcs.2019.08.027,"LATTES,SCOPUS"


## 9. Cruzamento de Periódicos com a Base de Percentil Scopus

Mesma lógica em camadas da versão anterior deste notebook (match exato por
nome, match exato por ISSN, depois busca bidirecional por substring apenas
no que sobrou), agora aplicada sobre `df_periodicos_unificado` em vez de
`df_bib_artigos` isolado. A única mudança estrutural é a chave de nome
usada para o match: `COALESCE(titulo_revista_lattes, titulo_revista_scopus)`
— assim um artigo de origem Scopus (que já chega com `titulo_revista_scopus`
preenchido, mas sem `titulo_revista_lattes`) participa do mesmo cruzamento
que um artigo de origem Lattes ou ORCID.

Cada linha é identificada pela `chave_dedup` (calculada na Seção 8) em vez
de `titulo + id_lattes` como na versão anterior — mesma ideia, mas usando a
chave que já garante uma linha por publicação real.

> **Nota de fidelidade ao comportamento original:** quando tanto o `issn` da
> base unificada quanto `E-ISSN`/`Print ISSN` (Scopus) são nulos, o
> `pd.merge` do pandas trata os dois nulos como iguais e gera um match "por
> ISSN" mesmo sem nenhum ISSN de fato existir nos dois lados. Esse
> comportamento já existia no notebook original e é preservado aqui.


In [18]:
def formatar_issn(issn):
    """Normaliza um ISSN para 8 dígitos sem hífen, retornando nulo se o valor for vazio/inválido."""
    issn_str = str(issn).replace('-', '').strip()
    if issn_str.lower() in ['nan', 'none', '', 'nat']:
        return pd.NA
    return issn_str.zfill(8)


COLUNAS_PERIODICO_UNIFICADO = [c for c in COLUNAS_PERIODICO if c != 'fonte'] + ['fontes', 'chave_dedup']

print("Carregando e preparando a base completa da Scopus...")
df_scopus_raw = pd.read_excel(ARQUIVO_PERCENTIL_SCOPUS)
df_scopus_raw['Title'] = df_scopus_raw['Title'].astype(str).str.upper().str.strip()
df_scopus_raw['E-ISSN'] = df_scopus_raw['E-ISSN'].apply(formatar_issn)
df_scopus_raw['Print ISSN'] = df_scopus_raw['Print ISSN'].apply(formatar_issn)

mask_comput = df_scopus_raw['Scopus Sub-Subject Area'].str.contains('Comput', case=False, na=False)
titulos_computacao = set(df_scopus_raw[mask_comput]['Title'].unique())
issns_computacao = set(df_scopus_raw[mask_comput]['E-ISSN'].dropna().unique()).union(
                    set(df_scopus_raw[mask_comput]['Print ISSN'].dropna().unique()))

# Um mesmo periódico pode aparecer várias vezes na planilha (uma linha por
# subárea ASJC). Mantemos apenas o percentil mais alto de cada título.
df_scopus_unicos = df_scopus_raw.sort_values(by='Percentile', ascending=False)
df_scopus_unicos = df_scopus_unicos.drop_duplicates(subset=['Title'], keep='first').copy()

colunas_scopus = [
    'Scopus Source ID', 'Title', 'Percentile',
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 'Print ISSN'
]
df_scopus_filtro = df_scopus_unicos[colunas_scopus]

print("Preparando a base unificada de periódicos (chave de cruzamento)...")
df_base_periodicos = df_periodicos_unificado.copy()
df_base_periodicos['revista'] = (
    df_base_periodicos['titulo_revista_lattes']
    .where(df_base_periodicos['titulo_revista_lattes'].notna(), df_base_periodicos['titulo_revista_scopus'])
    .astype(str).str.upper().str.strip()
)
df_base_periodicos['issn'] = df_base_periodicos['issn'].apply(formatar_issn)

print("Preparação concluída.")

Carregando e preparando a base completa da Scopus...


Preparando a base unificada de periódicos (chave de cruzamento)...
Preparação concluída.


### 9.1 Camadas 1 e 2 — Match exato (nome da revista e ISSN)

In [19]:
print("Realizando o cruzamento exato (ISSN e Nome Exato)...")

df_match_nome = pd.merge(df_base_periodicos, df_scopus_filtro, left_on='revista', right_on='Title', how='inner')
df_match_e_issn = pd.merge(df_base_periodicos, df_scopus_filtro, left_on='issn', right_on='E-ISSN', how='inner')
df_match_print_issn = pd.merge(df_base_periodicos, df_scopus_filtro, left_on='issn', right_on='Print ISSN', how='inner')

df_sucessos = pd.concat([df_match_nome, df_match_e_issn, df_match_print_issn], ignore_index=True)
df_sucessos = df_sucessos.drop_duplicates(subset=['chave_dedup']).copy()

df_sucessos['Computation Area'] = (
    df_sucessos['Title'].isin(titulos_computacao) |
    df_sucessos['E-ISSN'].isin(issns_computacao) |
    df_sucessos['Print ISSN'].isin(issns_computacao)
)

print(f"Matches exatos encontrados: {len(df_sucessos)}")

Realizando o cruzamento exato (ISSN e Nome Exato)...


Matches exatos encontrados: 1468


### 9.2 Camada 3 — Match bidirecional por substring (apenas no restante)

In [20]:
print("Busca bidirecional (nome contido) para os artigos restantes...")

chaves_com_match_exato = set(df_sucessos['chave_dedup'])
df_restante = df_base_periodicos[~df_base_periodicos['chave_dedup'].isin(chaves_com_match_exato)].copy()

# Ordena a lista de periódicos Scopus do nome mais longo para o mais curto,
# para evitar que um nome curto "roube" o match de um nome mais específico.
df_scopus_filtro_sorted = df_scopus_filtro.copy()
df_scopus_filtro_sorted['tamanho_titulo'] = df_scopus_filtro_sorted['Title'].str.len()
df_scopus_filtro_sorted = df_scopus_filtro_sorted.sort_values(by='tamanho_titulo', ascending=False)
lista_scopus = df_scopus_filtro_sorted.to_dict('records')


def busca_bidirecional_revista(revista_lattes):
    """Procura, na lista Scopus, um título que contenha (ou esteja contido em) o nome da revista."""
    if pd.isna(revista_lattes) or revista_lattes == 'NAN' or revista_lattes == '':
        return None

    for scopus in lista_scopus:
        titulo_scopus = scopus['Title']
        if pd.notna(titulo_scopus) and titulo_scopus != 'NAN' and titulo_scopus != "":
            if (titulo_scopus in revista_lattes) or (revista_lattes in titulo_scopus):
                return scopus
    return None


resultados_parciais = df_restante['revista'].apply(busca_bidirecional_revista)

mask_encontrados = resultados_parciais.notna()
df_match_parcial = df_restante[mask_encontrados].copy()

if not df_match_parcial.empty:
    dicts_encontrados = resultados_parciais[mask_encontrados]
    for col in colunas_scopus:
        df_match_parcial[col] = [d[col] for d in dicts_encontrados]

    df_match_parcial['Computation Area'] = (
        df_match_parcial['Title'].isin(titulos_computacao) |
        df_match_parcial['E-ISSN'].isin(issns_computacao) |
        df_match_parcial['Print ISSN'].isin(issns_computacao)
    )

    df_sucessos = pd.concat([df_sucessos, df_match_parcial], ignore_index=True)
    df_sucessos = df_sucessos.drop_duplicates(subset=['chave_dedup']).copy()

print(f"Total de sucessos após a busca bidirecional: {len(df_sucessos)}")

Busca bidirecional (nome contido) para os artigos restantes...


Total de sucessos após a busca bidirecional: 1682


### 9.3 Falhas de match e consolidação final

In [21]:
print("Isolando e tratando as falhas definitivas...")

df_falhas = df_restante[~mask_encontrados].copy()
df_falhas['Scopus Source ID'] = pd.NA
df_falhas['Title'] = pd.NA
df_falhas['Percentile'] = 0
df_falhas['Scopus ASJC Code (Sub-subject Area)'] = pd.NA
df_falhas['Scopus Sub-Subject Area'] = pd.NA
df_falhas['E-ISSN'] = pd.NA
df_falhas['Print ISSN'] = pd.NA
df_falhas['Computation Area'] = False

print("Consolidando o resultado do cruzamento...")
df_periodicos_tratado = pd.concat([df_sucessos, df_falhas], ignore_index=True)
df_periodicos_tratado['Percentile'] = df_periodicos_tratado['Percentile'].astype(int)
df_periodicos_tratado['Computation Area'] = df_periodicos_tratado['Computation Area'].astype(bool)

# 'match_adequado' indica se o artigo encontrou correspondência válida na Scopus
df_periodicos_tratado['match_adequado'] = df_periodicos_tratado['Scopus Source ID'].notna()

# Sobrescreve os campos de cruzamento com o resultado, mantendo os demais
# campos (id_lattes, titulo_artigo, ano_pub, doi, autores, fontes, chave_dedup) intactos.
df_periodicos_tratado['id_scopus'] = df_periodicos_tratado['Scopus Source ID']
df_periodicos_tratado['titulo_revista_scopus'] = df_periodicos_tratado['Title']
df_periodicos_tratado['maior_percentil'] = df_periodicos_tratado['Percentile']
df_periodicos_tratado['codigo_area_maior_percentil'] = df_periodicos_tratado['Scopus ASJC Code (Sub-subject Area)']
df_periodicos_tratado['area_maior_percentil'] = df_periodicos_tratado['Scopus Sub-Subject Area']
df_periodicos_tratado['issn'] = df_periodicos_tratado['E-ISSN']
df_periodicos_tratado['computation_area'] = df_periodicos_tratado['Computation Area']

df_periodicos_unificado = df_periodicos_tratado[COLUNAS_PERIODICO_UNIFICADO].copy()

print(f"Tabela unificada de periódicos tratada. Total de linhas: {len(df_periodicos_unificado)}")
display(df_periodicos_unificado.sample(min(10, len(df_periodicos_unificado))))

Isolando e tratando as falhas definitivas...
Consolidando o resultado do cruzamento...
Tabela unificada de periódicos tratada. Total de linhas: 1746


,id_lattes,titulo_artigo,titulo_revista_lattes,ano_pub,doi,autores,match_adequado,coautoria_aluno,id_scopus,titulo_revista_scopus,maior_percentil,codigo_area_maior_percentil,area_maior_percentil,issn,computation_area,fontes,chave_dedup
1147,2718664296804955,Adverse perinatal outcomes for advanced matern...,NaN,2015.0,10.1016/j.jpedp.2015.07.004,NaN,True,NaN,29346,SOUTH ATLANTIC QUARTERLY,99,1208,Literature and Literary Theory,NaN,False,ORCID,2718664296804955|DOI:10.1016/j.jpedp.2015.07.004
321,2291334095539768,A multi-criteria approach to support frequency...,TRANSPORTATION RESEARCH PROCEDIA,2025.0,http://dx.doi.org/10.1016/j.trpro.2025.04.092,"CAETANO, J. A. ; SOUSA, J. P. ; MARQUES, C. M....",True,NaN,21100448300,TRANSPORTATION RESEARCH PROCEDIA,43,3313,Transportation,23521465,False,"LATTES,SCOPUS",2291334095539768|DOI:10.1016/j.trpro.2025.04.092
752,9521646119786469,Combining integer linear programming with a st...,INTERNATIONAL TRANSACTIONS IN OPERATIONAL RESE...,2019.0,http://dx.doi.org/10.1111/itor.12563,"CLÍMACO, GLAUBOS ; ROSSETI, ISABEL ; Simonetti...",True,NaN,9700153238,INTERNATIONAL TRANSACTIONS IN OPERATIONAL RESE...,80,1403,Business and International Management,14753995,True,"LATTES,ORCID,SCOPUS",9521646119786469|DOI:10.1111/itor.12563
87,2002515486942024,Disconnected Matchings,THEORETICAL COMPUTER SCIENCE,2023.0,10.1016/j.tcs.2023.113821,"GOMES, G. C. M. ; MASQUIO, B. P. ; PINTO, P. E...",True,NaN,20571,THEORETICAL COMPUTER SCIENCE,44,2614,Theoretical Computer Science,NaN,True,"LATTES,SCOPUS",2002515486942024|DOI:10.1007/978-3-030-89543-3_48
1073,4436183480921146,Uma heurística para o problema de programação ...,TENDÊNCIAS EM MATEMÁTICA APLICADA E COMPUTACIONAL,2001.0,NaN,"SOUZA, M. J. F. ; MACULAN FILHO, N. ; OCHI, Lu...",True,NaN,29346,SOUTH ATLANTIC QUARTERLY,99,1208,Literature and Literary Theory,NaN,False,LATTES,4436183480921146|TIT:UMA HEURISTICA PARA O PRO...
374,3957046121364560,A reversible circuit synthesis algorithm with ...,JOURNAL OF UNIVERSAL COMPUTER SCIENCE,2021.0,http://dx.doi.org/10.3897/jucs.69617,"DALCUMUNE, E. ; KOWADA, LUIS ANTONIO B. ; Ribe...",True,NaN,145537,JOURNAL OF UNIVERSAL COMPUTER SCIENCE,42,2614,Theoretical Computer Science,09486968,True,"LATTES,SCOPUS",3957046121364560|DOI:10.3897/jucs.69617
630,4436183480921146,A Smoothing Optimization Approach Applied to t...,IEEE LATIN AMERICA TRANSACTIONS,2020.0,NaN,"XAVIER, VINÍCIUS LAYTER ; Maculan, N. ; Pessan...",True,NaN,19700181218,IEEE LATIN AMERICA TRANSACTIONS,62,2208,Electrical and Electronic Engineering,15480992,True,LATTES,4436183480921146|TIT:A SMOOTHING OPTIMIZATION ...
31,6243465206463403,On a multisensor knowledge fusion heuristic fo...,COMPUTER COMMUNICATIONS,2021.0,http://dx.doi.org/10.1016/j.comcom.2021.04.025,"MARTINS, GABRIEL ; DE SOUZA, SERGIO GUEDES ; L...",True,NaN,13681,COMPUTER COMMUNICATIONS,91,1705,Computer Networks and Communications,1873703X,True,"LATTES,SCOPUS",6243465206463403|DOI:10.1016/j.comcom.2021.04.025
1292,3957046121364560,The perfection and recognition of bull-reducib...,RAIRO. INFORMATIQUE THÉORIQUE ET APPLICATIONS,2005.0,http://dx.doi.org/10.1051/ita:2005009,"EVERETT, H. ; FIGUEIREDO, C. M. H. ; KLEIN, S....",True,NaN,13012,RAIRO - THEORETICAL INFORMATICS AND APPLICATIONS,24,2600,Mathematics (all),1290385X,True,"LATTES,ORCID,SCOPUS",3957046121364560|DOI:10.1051/ita:2005009
1466,3621433615334969,A study of networks simulation efficiency: Flu...,NaN,2001.0,10.1109/INFCOM.2001.916619,"Liu, Benyuan;Figueiredo, Daniel R.;Guo, Yang;K...",True,NaN,18204,PROCEEDINGS - IEEE INFOCOM,89,2208,Electrical and Electronic Engineering,NaN,True,"ORCID,SCOPUS",3621433615334969|DOI:10.1109/infcom.2001.916619


## 10. Cruzamento de Trabalhos de Congresso com a Base de Eventos Classificados

Mesma lógica de match fuzzy (`rapidfuzz`) da versão anterior, agora sobre
`df_congressos_unificado`. Como o schema final já usa `titulo_evento_lattes`
como nome do campo de evento (preenchido a partir do nome do periódico/evento
retornado por qualquer uma das três fontes — não só o Lattes), o cruzamento
funciona igual para trabalhos vindos de ORCID/Scopus, sem precisar de nenhuma
adaptação de nome de coluna.


### 10.1 Carga e padronização da base de eventos classificados (Google/CAPES)

In [22]:
print("Carregando a base de eventos classificados...")
df_google_raw = pd.read_csv(ARQUIVO_EVENTOS_CLASSIFICADOS)

print("Distribuição original de estratos:")
display(df_google_raw['Estrato'].value_counts())

Carregando a base de eventos classificados...
Distribuição original de estratos:


Estrato
A3    171
A4    134
A1    110
B4     90
A2     86
B1     78
B2     60
B3     52
Name: count, dtype: int64

In [23]:
# A base de eventos usa rótulos B1-B4 para os estratos mais baixos; o projeto
# usa a faixa estendida A1-A8, então B1-B4 são remapeados para A5-A8.
print("Remapeando estratos B1-B4 -> A5-A8...")

mapeamento_estratos = {
    'B1': 'A5',
    'B2': 'A6',
    'B3': 'A7',
    'B4': 'A8'
}
df_google_raw['Estrato'] = df_google_raw['Estrato'].replace(mapeamento_estratos)

print("Nova distribuição de estratos:")
display(df_google_raw['Estrato'].value_counts())

# Padroniza o nome do evento em maiúsculas
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper()

Remapeando estratos B1-B4 -> A5-A8...
Nova distribuição de estratos:


Estrato
A3    171
A4    134
A1    110
A8     90
A2     86
A5     78
A6     60
A7     52
Name: count, dtype: int64

### 10.2 Normalização de texto (remoção de acentos, maiúsculas, espaços)

In [24]:
print("Normalizando nomes de eventos (acentos, maiúsculas, espaços) em ambas as bases...")


def limpar_texto(serie):
    """Remove acentos, converte para maiúsculas, colapsa espaços múltiplos e tira espaços nas bordas."""
    return (serie.astype(str)
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.upper()
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip())


df_base_congressos = df_congressos_unificado.copy()
df_base_congressos['evento_limpo'] = limpar_texto(df_base_congressos['titulo_evento_lattes'])
df_google_raw['Nome do evento'] = limpar_texto(df_google_raw['Nome do evento'])
df_google_raw['Nome do evento em inglês'] = limpar_texto(df_google_raw['Nome do evento em inglês'])
df_google_raw['Sigla'] = limpar_texto(df_google_raw['Sigla'])

# Para a busca por substring/sigla funcionar bem nos dois idiomas, ordenamos a
# base de eventos pelo maior nome disponível entre PT e EN. Isso evita que um
# nome curto "roube" o match de um nome mais longo e específico.
df_google_raw['tamanho_pt'] = df_google_raw['Nome do evento'].str.len()
df_google_raw['tamanho_en'] = df_google_raw['Nome do evento em inglês'].str.len()
df_google_raw['tamanho_max'] = df_google_raw[['tamanho_pt', 'tamanho_en']].max(axis=1)
df_google_raw = df_google_raw.sort_values(by='tamanho_max', ascending=False)

# Lista de dicionários — acesso mais rápido do que iterrows() em um DataFrame
lista_google = df_google_raw.to_dict('records')

print("Normalização concluída.")

Normalizando nomes de eventos (acentos, maiúsculas, espaços) em ambas as bases...
Normalização concluída.


### 10.3 Função de match fuzzy (sigla exata > similaridade textual PT/EN)

A função tenta, em ordem de confiança:

1. **Sigla exata**, isolada por limites de palavra (regex `\b`) — score 100.
2. **Similaridade fuzzy** (`token_set_ratio`) contra o nome do evento em
   português e em inglês, mantendo o maior score entre os dois.

Só é considerado match válido se o melhor score atingir o limiar de corte
(`LIMIAR_CORTE_FUZZY = 95`, em uma escala de 0 a 100).


In [25]:
LIMIAR_CORTE_FUZZY = 95  # Escala 0-100; valor alto para evitar falsos positivos


def encontrar_melhor_match_fuzzy(evento_lattes):
    """Procura, na base de eventos classificados, o melhor match fuzzy para um nome de evento.

    Retorna uma tupla: (sigla, nome_do_evento_padronizado, estrato, tipo_match, score_confianca).
    Quando não há match acima do limiar, retorna estrato 'A8' (pior classificação) e tipo 'Sem Match'.
    """
    if pd.isna(evento_lattes) or evento_lattes == 'NAN' or evento_lattes == '':
        return pd.NA, pd.NA, 'A8', 'Sem Match', 0

    melhor_google_match = None
    maior_score_encontrado = 0
    tipo_do_melhor_match = 'Sem Match'

    # --- Tentativa 1: sigla exata isolada por limites de palavra ---
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla Exata', 100

    # --- Tentativa 2: similaridade fuzzy (token_set_ratio) em PT e EN ---
    for google in lista_google:
        nome_pt = google['Nome do evento']
        nome_en = google['Nome do evento em inglês']

        score_pt = 0
        score_en = 0

        if pd.notna(nome_pt) and nome_pt != 'NAN' and nome_pt != "":
            score_pt = fuzz.token_set_ratio(evento_lattes, nome_pt)

        if pd.notna(nome_en) and nome_en != 'NAN' and nome_en != "":
            score_en = fuzz.token_set_ratio(evento_lattes, nome_en)

        score_atual_max = max(score_pt, score_en)

        if score_atual_max > maior_score_encontrado:
            maior_score_encontrado = score_atual_max
            melhor_google_match = google
            tipo_do_melhor_match = 'Fuzzy Nome PT' if score_pt >= score_en else 'Fuzzy Nome EN'

    # --- Decisão final: o melhor score supera o limiar de segurança? ---
    if maior_score_encontrado >= LIMIAR_CORTE_FUZZY:
        return (
            melhor_google_match['Sigla'],
            melhor_google_match['Nome do evento'],
            melhor_google_match['Estrato'],
            tipo_do_melhor_match,
            maior_score_encontrado
        )

    # Score insuficiente: rejeita o match para evitar falso positivo
    return pd.NA, pd.NA, 'A8', 'Sem Match', maior_score_encontrado

### 10.4 Aplicação do match e consolidação de `df_congressos_unificado`

In [26]:
print("Aplicando o match fuzzy a todos os trabalhos de congresso (pode levar alguns segundos)...")

resultados = df_base_congressos['evento_limpo'].apply(encontrar_melhor_match_fuzzy)

df_base_congressos['sigla_evento_google'] = [res[0] for res in resultados]
df_base_congressos['titulo_evento_google'] = [res[1] for res in resultados]
df_base_congressos['estrato'] = [res[2] for res in resultados]
df_base_congressos['tipo_match'] = [res[3] for res in resultados]
df_base_congressos['score_confianca'] = [res[4] for res in resultados]

# --- Relatório-resumo do cruzamento ---
total_originais = len(df_base_congressos)
qtd_sigla = (df_base_congressos['tipo_match'] == 'Por Sigla Exata').sum()
qtd_fuzzy_pt = (df_base_congressos['tipo_match'] == 'Fuzzy Nome PT').sum()
qtd_fuzzy_en = (df_base_congressos['tipo_match'] == 'Fuzzy Nome EN').sum()
qtd_falhas = (df_base_congressos['tipo_match'] == 'Sem Match').sum()

print("\n--- Relatório de Cruzamento de Eventos ---")
print(f"Total de trabalhos de congresso (unificado): {total_originais}")
if total_originais:
    print(f"Match por Sigla Exata: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
    print(f"Match Fuzzy (Nome PT):  {qtd_fuzzy_pt} ({round((qtd_fuzzy_pt/total_originais)*100, 1)}%)")
    print(f"Match Fuzzy (Nome EN):  {qtd_fuzzy_en} ({round((qtd_fuzzy_en/total_originais)*100, 1)}%)")
    print(f"Sem Match:              {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

    display(
        df_base_congressos[df_base_congressos['tipo_match'] != 'Sem Match']
        [['titulo_evento_lattes', 'titulo_evento_google', 'estrato', 'tipo_match', 'score_confianca']]
        .sample(min(5, total_originais))
    )

Aplicando o match fuzzy a todos os trabalhos de congresso (pode levar alguns segundos)...



--- Relatório de Cruzamento de Eventos ---
Total de trabalhos de congresso (unificado): 3428
Match por Sigla Exata: 1212 (35.4%)
Match Fuzzy (Nome PT):  105 (3.1%)
Match Fuzzy (Nome EN):  705 (20.6%)
Sem Match:              1406 (41.0%)


,titulo_evento_lattes,titulo_evento_google,estrato,tipo_match,score_confianca
3037,SIMPÓSIO BRASILEIRO DE BANCO DE DADOS,BRAZILIAN SYMPOSIUM ON DATABASES,A4,Fuzzy Nome EN,100.0
2621,INTERNATIONAL CONFERENCE ON VERY LARGE DATA BA...,CONFERENCIA INTERNACIONAL SOBRE CIENCIA DE DAD...,A1,Por Sigla Exata,100.0
72,SEMINÁRIO INTEGRADO DE SOFTWARE E HARDWARE,INTEGRATED SEMINAR ON SOFTWARE AND HARDWARE,A4,Fuzzy Nome EN,100.0
1836,SOFTWARE ENGINEERING AND ADVANCED APPLICATIONS,CONFERENCIA EUROMICRO SOBRE ENGENHARIA DE SOFT...,A3,Fuzzy Nome EN,100.0
161,2017 IEEE 14TH INTERNATIONAL CONFERENCE ON NET...,CONFERENCIA DE REDES IFIP,A3,Por Sigla Exata,100.0


### 10.5 Auditoria da "zona crítica" de confiança

Matches com score entre o limiar mínimo (95) e quase-perfeito (99) merecem
uma segunda olhada manual antes de confiar 100% no resultado.


In [27]:
limiar_inferior = 95
limiar_superior = 99

if not df_base_congressos.empty:
    df_zona_critica = df_base_congressos[
        (df_base_congressos['score_confianca'] >= limiar_inferior) &
        (df_base_congressos['score_confianca'] <= limiar_superior)
    ].copy()

    df_zona_critica = df_zona_critica.sort_values(by='score_confianca', ascending=True)

    colunas_para_auditoria = ['titulo_evento_lattes', 'titulo_evento_google', 'sigla_evento_google', 'estrato', 'tipo_match', 'score_confianca']
    tabela_auditoria = df_zona_critica[colunas_para_auditoria]

    print(f"Encontrados {len(tabela_auditoria)} registros na zona crítica (score {limiar_inferior} a {limiar_superior}).")
    print("Recomenda-se leitura atenta para garantir que não há homônimos.\n")

    display(
        tabela_auditoria.style.background_gradient(
            subset=['score_confianca'],
            cmap='YlOrRd_r',
            vmin=limiar_inferior,
            vmax=limiar_superior
        )
    )

Encontrados 106 registros na zona crítica (score 95 a 99).
Recomenda-se leitura atenta para garantir que não há homônimos.



,titulo_evento_lattes,titulo_evento_google,sigla_evento_google,estrato,tipo_match,score_confianca
3169,Proceedings of the 2011 5th Ftra International Conference on Multimedia and Ubiquitous Engineering Mue 2011,CONFERENCIA INTERNACIONAL ACM SOBRE MULTIMIDIA,ACMMM,A1,Fuzzy Nome EN,95.000000
386,XIX SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1023,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1017,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1025,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1026,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1014,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1024,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1016,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1232,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951


### 10.6 Consolidação final de `df_congressos_unificado`

In [28]:
COLUNAS_CONGRESSO_UNIFICADO = [c for c in COLUNAS_CONGRESSO if c != 'fonte'] + ['fontes', 'chave_dedup']

df_congressos_unificado = df_base_congressos[COLUNAS_CONGRESSO_UNIFICADO].copy()

print("Estrutura final de df_congressos_unificado:")
df_congressos_unificado.info()

Estrutura final de df_congressos_unificado:
<class 'pandas.DataFrame'>
RangeIndex: 3428 entries, 0 to 3427
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_lattes             3428 non-null   str    
 1   titulo_artigo         3428 non-null   str    
 2   ano                   3428 non-null   Float64
 3   doi                   1427 non-null   str    
 4   autores               3398 non-null   object 
 5   titulo_evento_lattes  3391 non-null   object 
 6   paginas               2505 non-null   object 
 7   sigla_evento_google   2022 non-null   str    
 8   titulo_evento_google  2022 non-null   str    
 9   estrato               3428 non-null   str    
 10  tipo_match            3428 non-null   str    
 11  coautoria_aluno       0 non-null      object 
 12  fontes                3428 non-null   str    
 13  chave_dedup           3428 non-null   object 
dtypes: Float64(1), object(5), str(8)
memory

## 11. Detecção de Coautoria de Alunos nas Produções

Mesma lógica da versão anterior deste notebook, aplicada a
`df_periodicos_unificado` e `df_congressos_unificado`. A estratégia
permanece: gerar exaustivamente todas as variações plausíveis do nome de
cada aluno e verificar se alguma delas aparece como substring isolada na
string de autores de cada publicação.

> A cobertura desta etapa depende de `autores` estar preenchido — para
> publicações de origem ORCID isso normalmente não acontece (ver
> limitação na Seção 6), então a coautoria de aluno tende a só ser
> detectada em publicações de origem Lattes ou Scopus.


### 11.1 Normalização de texto e geração de variações de nome

In [29]:
def normalizar_texto(valor):
    """Remove acentos, força maiúsculas e reduz qualquer caractere não alfanumérico a um único espaço."""
    if pd.isna(valor):
        return ""

    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', errors='ignore').decode('utf-8')
    texto = texto.upper()
    texto = re.sub(r'[^A-Z0-9]+', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()


def gerar_todas_abreviacoes(nome_completo_norm):
    """Gera o conjunto de variações plausíveis de citação acadêmica de um nome completo normalizado.

    Cobre os padrões mais comuns na autoria brasileira: uso do sobrenome
    materno/intermediário como "sobrenome de citação", sobrenomes compostos,
    nomes de batismo duplos e a inversão Sobrenome-Nome / Nome-Sobrenome.
    """
    preposicoes = {'DE', 'DA', 'DO', 'DAS', 'DOS', 'E'}
    partes_originais = nome_completo_norm.split()
    partes_uteis = [p for p in partes_originais if p not in preposicoes]

    if len(partes_uteis) < 2:
        return {nome_completo_norm}

    variacoes = set([nome_completo_norm])

    # --- Define os possíveis blocos de "sobrenome de citação" ---
    sobrenomes_alvo = [partes_uteis[-1]]  # último nome (ex.: "CARNEIRO")

    # Qualquer nome do meio também pode ser o sobrenome de citação
    for i in range(1, len(partes_uteis) - 1):
        sobrenomes_alvo.append(partes_uteis[i])

    # Combinação composta dos dois últimos nomes (ex.: "DIAS CARNEIRO")
    if len(partes_uteis) >= 3:
        sobrenomes_alvo.append(f"{partes_uteis[-2]} {partes_uteis[-1]}")

    # Com a preposição original, se existir (ex.: "DE CARNEIRO")
    if len(partes_originais) >= 2 and partes_originais[-2] in preposicoes:
        sobrenomes_alvo.append(f"{partes_originais[-2]} {partes_originais[-1]}")

    # --- Combina cada sobrenome candidato com as iniciais do nome restante ---
    for sobrenome in set(sobrenomes_alvo):
        sobrenome_partes = sobrenome.split()
        resto = [p for p in partes_uteis if p not in sobrenome_partes]

        if not resto:
            continue

        iniciais = [p[0] for p in resto]
        primeiro_nome = resto[0]
        inicial_primeira = iniciais[0]

        iniciais_com_espaco = " ".join(iniciais)
        iniciais_sem_espaco = "".join(iniciais)
        duas_iniciais = f"{iniciais[0]} {iniciais[1]}" if len(iniciais) > 1 else inicial_primeira

        primeiro_mais_iniciais = primeiro_nome
        if len(iniciais) > 1:
            primeiro_mais_iniciais += " " + " ".join(iniciais[1:])

        # Possíveis blocos do "nome de batismo" (a parte antes do sobrenome)
        blocos_nome = [
            iniciais_com_espaco,       # ex.: "J V D C"
            iniciais_sem_espaco,       # ex.: "JVDC"
            inicial_primeira,          # ex.: "J"
            duas_iniciais,             # ex.: "J V"
            primeiro_nome,             # ex.: "JOAO"
            primeiro_mais_iniciais,    # ex.: "JOAO V D C"
            " ".join(resto)            # ex.: "JOAO VITOR DIAS CARNEIRO"
        ]

        # Permutação da ordem: Sobrenome-Nome e Nome-Sobrenome
        for bloco in set(blocos_nome):
            variacoes.add(f"{sobrenome} {bloco}")
            variacoes.add(f"{bloco} {sobrenome}")

    return variacoes

### 11.2 Extração dos alunos a partir dos JSONs brutos

In [30]:
def _selecionar_arquivo_mais_recente_por_aluno(diretorio_alunos):
    """Alguns alunos têm mais de um JSON raspado na pasta (re-raspagem em
    datas diferentes); escolhe o arquivo com 'atualizacao_cv' mais recente
    por id_lattes, para que a escolha não dependa da ordem arbitrária do
    glob. Reaproveitado por extrair_alunos (Seção 11.2) e por
    extrair_formacao_aluno (Seção 11.2.2)."""
    pasta = Path(diretorio_alunos)
    if not pasta.exists() or not pasta.is_dir():
        return {}

    escolhidos = {}  # id_lattes -> (atualizacao_cv_dt, caminho)
    for arquivo_json in sorted(pasta.glob('*.json')):
        try:
            with open(arquivo_json, 'r', encoding='utf-8') as f:
                dados = json.load(f)
        except (json.JSONDecodeError, OSError) as erro:
            print(f"Erro ao ler {arquivo_json}: {erro}")
            continue

        info = dados.get('informacoes_pessoais', {})
        id_lattes = info.get('id_lattes')
        if not id_lattes:
            continue

        atualizacao_dt = pd.to_datetime(
            str(info.get('atualizacao_cv', '')).strip(), format='%d/%m/%Y', errors='coerce'
        )

        atual = escolhidos.get(id_lattes)
        if atual is None or (pd.notna(atualizacao_dt) and (pd.isna(atual[0]) or atualizacao_dt >= atual[0])):
            escolhidos[id_lattes] = (atualizacao_dt, arquivo_json)

    return {id_lattes: caminho for id_lattes, (_, caminho) in escolhidos.items()}


def extrair_alunos(diretorio_alunos):
    """Lê os JSONs brutos dos alunos (o mais recente por id_lattes, ver
    _selecionar_arquivo_mais_recente_por_aluno) e monta um DataFrame com
    todas as variações de nome geradas."""
    registros = []
    arquivos_por_aluno = _selecionar_arquivo_mais_recente_por_aluno(diretorio_alunos)

    if not arquivos_por_aluno:
        pasta = Path(diretorio_alunos)
        if not pasta.exists() or not pasta.is_dir():
            print(f"AVISO: diretório de alunos não encontrado: {diretorio_alunos}")
        return pd.DataFrame()

    for id_lattes, arquivo_json in sorted(arquivos_por_aluno.items()):
        try:
            with open(arquivo_json, 'r', encoding='utf-8') as f:
                dados_aluno = json.load(f)

            info = dados_aluno.get('informacoes_pessoais', {})
            nome_completo = info.get('nome_completo', '')
            nome_citacoes = info.get('nome_citacoes', '')

            if not nome_completo:
                continue

            # Variações de citação que o próprio Lattes do aluno já declara
            citacoes = [item.strip() for item in str(nome_citacoes).split(';') if item.strip()]

            nome_comp_norm = normalizar_texto(nome_completo)
            todas_permutacoes = gerar_todas_abreviacoes(nome_comp_norm)

            variacoes_normalizadas = list(todas_permutacoes)
            for variacao in citacoes:
                nome_norm_citacao = normalizar_texto(variacao)
                if nome_norm_citacao and nome_norm_citacao not in variacoes_normalizadas:
                    variacoes_normalizadas.append(nome_norm_citacao)

            registros.append({
                'id_lattes': str(id_lattes),
                'nome_completo': str(nome_completo).strip(),
                'nome_citacoes': str(nome_citacoes).strip(),
                'nome_completo_normalizado': nome_comp_norm,
                'nome_citacoes_normalizadas': ' | '.join(variacoes_normalizadas),
                'variacoes_coautoria': ' | '.join(variacoes_normalizadas),
            })
        except (json.JSONDecodeError, OSError) as erro:
            print(f"Erro ao ler {arquivo_json}: {erro}")

    df_alunos_local = pd.DataFrame(registros)
    if not df_alunos_local.empty:
        df_alunos_local = df_alunos_local.drop_duplicates(subset=['id_lattes']).copy()

    return df_alunos_local


print("Extraindo dados de alunos...")
df_alunos = extrair_alunos(CAMINHO_PASTA_ALUNOS)
print(f"Total de alunos carregados: {len(df_alunos)}")
display(df_alunos.head())

Extraindo dados de alunos...


Total de alunos carregados: 297


,id_lattes,nome_completo,nome_citacoes,nome_completo_normalizado,nome_citacoes_normalizadas,variacoes_coautoria
0,1766965412894981,João Luís da Silva Guio Soares,"SOARES, J. L. S. G.;GUIO, J. L.",JOAO LUIS DA SILVA GUIO SOARES,SILVA JOAO | GUIO SOARES JOAO L S | JOAO GUIO ...,SILVA JOAO | GUIO SOARES JOAO L S | JOAO GUIO ...
1,0352188533423371,David Ventura Cardoso,"CARDOSO, D. V.",DAVID VENTURA CARDOSO,VENTURA DC | VENTURA D C | CARDOSO DV | DAVID ...,VENTURA DC | VENTURA D C | CARDOSO DV | DAVID ...
2,8773283315440616,Ana Clara Correa da Silva,"SILVA, A. C. C.",ANA CLARA CORREA DA SILVA,CLARA ACS | A C CLARA | DA SILVA ANA | DA SILV...,CLARA ACS | A C CLARA | DA SILVA ANA | DA SILV...
3,6910314996365495,Fabio Luiz Silva Nogueira,"NOGUEIRA, F. L. S.",FABIO LUIZ SILVA NOGUEIRA,LUIZ F S N | FLS NOGUEIRA | F SILVA | NOGUEIRA...,LUIZ F S N | FLS NOGUEIRA | F SILVA | NOGUEIRA...
4,2636706873331793,Felipe Bevilaqua Foldes Guimarães,"GUIMARÃES, F. B. F.;GUIMARÃES, FELIPE BEVILAQU...",FELIPE BEVILAQUA FOLDES GUIMARAES,BEVILAQUA FELIPE F G | BEVILAQUA F | FELIPE GU...,BEVILAQUA FELIPE F G | BEVILAQUA F | FELIPE GU...


### 11.2.1 Resolução de nome de orientador → `id_lattes` de professor

Extraído da antiga Seção 14 para virar uma função compartilhada: tanto a
Seção 11.2.2 (formação acadêmica declarada pelo próprio aluno, abaixo)
quanto a Seção 14 (planilha administrativa) precisam resolver um nome de
orientador em texto livre para o `id_lattes` de um professor -- é o mesmo
problema nas duas fontes, então a lógica (e o alias manual em
`alias_orientadores.csv`) mora aqui uma única vez.

Sempre exata (conjunto de tokens do nome, sem preposições, nunca fuzzy) --
mesmo princípio de [[feedback_dedup_por_professor]].

In [ ]:
PREPOSICOES_NOME = {'DE', 'DA', 'DO', 'DAS', 'DOS', 'E'}


def tokens_nome(nome):
    """Tokeniza um nome normalizado, descartando preposições -- usado para
    casar um nome curto (ex.: 'Franklin Marquezino') contra o nome completo
    de um professor (ex.: 'Franklin de Lima Marquezino') por inclusão exata
    de conjuntos, sem nenhuma medida de similaridade/fuzzy."""
    return {p for p in normalizar_texto(nome).split() if p not in PREPOSICOES_NOME}


df_professores_tok = df_pessoas[['id_lattes', 'nome_completo']].copy()
df_professores_tok['tokens'] = df_professores_tok['nome_completo'].apply(tokens_nome)

# Alias manual para nomes de orientador que não batem automaticamente
# (typo, abreviação, coorientador externo). Arquivo opcional -- se não
# existir ou não tiver a linha, simplesmente não há alias para aquele nome.
df_alias_orientadores = pd.DataFrame(columns=['nome_no_xlsx', 'id_lattes'])
if Path(CAMINHO_ALIAS_ORIENTADORES).exists():
    df_alias_orientadores = pd.read_csv(CAMINHO_ALIAS_ORIENTADORES, dtype=str).dropna(subset=['nome_no_xlsx', 'id_lattes'])
mapa_alias_orientadores = dict(zip(
    df_alias_orientadores['nome_no_xlsx'].map(normalizar_texto),
    df_alias_orientadores['id_lattes'],
))


def resolver_id_professor(nome_orientador):
    """Resolve um nome de orientador (texto livre -- planilha administrativa
    ou formação acadêmica declarada pelo aluno) para um único id_lattes de
    professor, via alias manual ou inclusão exata de tokens. Retorna None
    quando não há alias e a busca por tokens não resulta em exatamente um
    candidato (nenhum, ou mais de um -- ambíguo demais para decidir sozinho).
    Reaproveitado pela Seção 11.2.2 e pela Seção 14."""
    alias = mapa_alias_orientadores.get(normalizar_texto(nome_orientador))
    if alias:
        return alias

    tok_orientador = tokens_nome(nome_orientador)
    candidatos = df_professores_tok[df_professores_tok['tokens'].apply(tok_orientador.issubset)]
    if len(candidatos) == 1:
        return candidatos.iloc[0]['id_lattes']
    return None


print("Função resolver_id_professor pronta (reaproveitada pela Seção 11.2.2 e pela Seção 14).")

### 11.2.2 Formação acadêmica declarada pelo próprio aluno

Terceira fonte para confirmar orientações, além da planilha administrativa
(Seção 14) e do Lattes do professor (`tb_orientacoes`): o próprio JSON do
aluno tem uma chave `formacao_academica` em que ELE declara, para cada grau
(Mestrado/Doutorado), o título do trabalho, o ano de obtenção e o nome do
orientador -- texto livre digitado pelo aluno, numa fonte independente do
que o professor digita no Lattes dele.

Diferente do casamento da Seção 11.2.3 (nome do aluno -> `id_lattes`, que já
mostrou ser frágil a typo), aqui o aluno já está 100% identificado pelo
próprio arquivo (`id_lattes` no nome do JSON) -- só o nome do ORIENTADOR
(texto livre) precisa ser resolvido, reaproveitando `resolver_id_professor`
da Seção 11.2.1.

Etapa exploratória por enquanto: `df_formacao_aluno` ainda **não** é
persistida nem cruzada com `tb_situacao_orientandos` -- é só extraída e
validada aqui, para revisão antes do próximo passo (triangular com as
outras duas fontes).

In [ ]:
def _nivel_a_partir_do_tipo(tipo):
    """Extrai 'Mestrado'/'Doutorado' do campo 'tipo' da formação acadêmica
    (ex.: 'Doutorado em andamento em Engenharia de Sistemas e Computação')."""
    n = normalizar_texto(tipo)
    if 'DOUTORADO' in n:
        return 'Doutorado'
    if 'MESTRADO' in n:
        return 'Mestrado'
    return None


def _status_a_partir_do_item(tipo, ano_conclusao):
    """Deriva o status a partir do próprio campo 'tipo' e de 'ano_conclusao'
    -- não precisa reler o texto livre, o Lattes já estrutura isso."""
    n = normalizar_texto(tipo)
    if str(ano_conclusao).strip():
        return 'Concluída'
    if 'INTERROMPID' in n:
        return 'Interrompida'
    if 'ANDAMENTO' in n:
        return 'Em Andamento'
    return None


def _extrair_orientador_e_titulo(descricao):
    """Extrai o texto do orientador e do título de trabalho a partir do texto
    livre de 'descricao'. Remove primeiro qualquer trecho entre parênteses
    (ex.: período sanduíche com orientador externo, para não confundir com o
    orientador principal) e usa a ÚLTIMA ocorrência de 'Orientador:' fora de
    parênteses -- a mais próxima do fim é a do próprio programa."""
    sem_parenteses = re.sub(r'\([^)]*\)', '', str(descricao))

    orientador_texto = None
    ocorrencias = re.findall(r'Orientador:\s*([^.]+)\.', sem_parenteses)
    if ocorrencias:
        orientador_texto = ocorrencias[-1].strip()

    titulo_trabalho = None
    m_titulo = re.search(
        r'T[ií]tulo:\s*(.*?)(?:,\s*Ano de [Oo]bten[cç][ãa]o:|\.\s*Orientador:|$)',
        sem_parenteses,
        re.IGNORECASE,
    )
    if m_titulo:
        titulo_trabalho = m_titulo.group(1).strip().rstrip(',').strip()

    return orientador_texto, titulo_trabalho


def extrair_formacao_aluno(diretorio_alunos):
    """Lê, para cada aluno, o histórico de 'formacao_academica' do próprio
    JSON (Mestrado/Doutorado) -- terceira fonte, independente da planilha
    administrativa e do Lattes do professor, para confirmar orientações.
    Reaproveita o mesmo arquivo mais recente por aluno escolhido na Seção
    11.2 (_selecionar_arquivo_mais_recente_por_aluno)."""
    arquivos_por_aluno = _selecionar_arquivo_mais_recente_por_aluno(diretorio_alunos)
    registros = []

    for id_lattes, arquivo_json in sorted(arquivos_por_aluno.items()):
        with open(arquivo_json, 'r', encoding='utf-8') as f:
            dados_aluno = json.load(f)

        for item in dados_aluno.get('formacao_academica', []):
            nivel = _nivel_a_partir_do_tipo(item.get('tipo', ''))
            if not nivel:
                continue

            orientador_texto, titulo_trabalho = _extrair_orientador_e_titulo(item.get('descricao', ''))

            registros.append({
                'id_lattes_aluno': str(id_lattes),
                'nivel': nivel,
                'status': _status_a_partir_do_item(item.get('tipo', ''), item.get('ano_conclusao', '')),
                'ano_inicio': item.get('ano_inicio') or None,
                'ano_conclusao': item.get('ano_conclusao') or None,
                'titulo_trabalho_aluno': titulo_trabalho,
                'orientador_texto_aluno': orientador_texto,
                'id_lattes_professor_via_aluno': (
                    resolver_id_professor(orientador_texto) if orientador_texto else None
                ),
            })

    return pd.DataFrame(registros, columns=[
        'id_lattes_aluno', 'nivel', 'status', 'ano_inicio', 'ano_conclusao',
        'titulo_trabalho_aluno', 'orientador_texto_aluno', 'id_lattes_professor_via_aluno',
    ])


print("Extraindo formação acadêmica declarada pelo próprio aluno (Mestrado/Doutorado)...")
df_formacao_aluno = extrair_formacao_aluno(CAMINHO_PASTA_ALUNOS)
print(f"Total de registros de formação (Mestrado/Doutorado, todos os alunos): {len(df_formacao_aluno)}")
print(f"Com orientador resolvido para um professor conhecido: {df_formacao_aluno['id_lattes_professor_via_aluno'].notna().sum()}")
print(f"Com orientador declarado mas NÃO resolvido (typo/abreviação/orientador externo): "
      f"{df_formacao_aluno['orientador_texto_aluno'].notna().sum() - df_formacao_aluno['id_lattes_professor_via_aluno'].notna().sum()}")

df_orientador_aluno_nao_localizado = (
    df_formacao_aluno[df_formacao_aluno['orientador_texto_aluno'].notna() & df_formacao_aluno['id_lattes_professor_via_aluno'].isna()]
    [['id_lattes_aluno', 'nivel', 'orientador_texto_aluno']]
    .drop_duplicates()
    .sort_values('orientador_texto_aluno')
)
if not df_orientador_aluno_nao_localizado.empty:
    print(
        f"AVISO: {len(df_orientador_aluno_nao_localizado)} orientador(es) declarado(s) pelo aluno não "
        f"localizado(s) em tb_professores -- o mesmo alias de '{CAMINHO_ALIAS_ORIENTADORES}' resolveria aqui também."
    )
    display(df_orientador_aluno_nao_localizado)

display(df_formacao_aluno.head(10))

# --- Validação pontual: os 3 casos já investigados nesta sessão ---
print("\nValidação nos casos já conhecidos:")
for id_lattes_aluno_teste, rotulo in [
    ('8394957969514296', 'Larissa Monteiro da Fonseca Galeno'),
    ('5358073887811044', 'Guilherme Adamatti Bridi'),
    ('6375174844597170', 'Vinicius Dalto do Nascimento'),
]:
    print(f"\n{rotulo} ({id_lattes_aluno_teste}):")
    display(df_formacao_aluno[df_formacao_aluno['id_lattes_aluno'] == id_lattes_aluno_teste])

### 11.2.3 Enriquecimento de `df_alunos` → histórico completo em `tb_aluno_titulos`

Cruza cada aluno (por **nome normalizado exato**, sem fuzzy, reaproveitando as
variações de citação da Seção 11.2) com o CSV de defesas do PESC
(`CAMINHO_CSV_DEFESAS`) e com `df_orientacoes` — de **todos os professores**,
não apenas o orientador oficial —, gerando `df_aluno_titulos`: uma linha por
`(aluno, nível, professor, ano, fonte)`.

Diferente da versão anterior deste notebook, **não se escolhe mais um único
"título vencedor" por aluno** (não é mais "Doutorado sempre vence sobre
Mestrado"). Um mesmo aluno pode ter, por exemplo, Mestrado concluído com um
professor e Doutorado em andamento com outro, e ambos os vínculos ficam
registrados. `tb_alunos.titulo`/`tb_alunos.data_defesa` deixam de existir —
todo o histórico de títulos passa a viver só em `tb_aluno_titulos`.

Esta tabela é a base tanto do relatório "Títulos dos alunos no Lattes de cada
professor" (`app.py`) quanto da Seção 14, que passa a reaproveitar este mesmo
cruzamento em vez de repeti-lo.

- **Fonte `csv_defesas`**: nível (`M`/`D`) e ano vêm da CSV oficial de defesas
  do PESC. Sem professor associado (a planilha não registra orientador).
- **Fonte `lattes_orientacoes`**: nível, status e ano de conclusão vêm de
  `df_orientacoes`; o professor é quem registrou aquela orientação no próprio
  currículo Lattes.


In [ ]:
# ---------------------------------------------------------------------------
# 11.2.3 Histórico completo de títulos por aluno: df_aluno_titulos (1:N)
# ---------------------------------------------------------------------------
# Cruzamento por NOME NORMALIZADO EXATO (sem fuzzy), usando as mesmas
# variações de citação já geradas na Seção 11.2 -- reaproveitado depois pela
# Seção 14, que só precisa consultar esta tabela em vez de refazer o match.
#
# Cada (aluno, nível, professor, ano, status, fonte) vira uma linha --
# diferente da versão anterior deste notebook, aqui NÃO se escolhe um único
# "título vencedor" por aluno: um mesmo aluno pode ter Mestrado com um
# professor e Doutorado com outro, por exemplo, e ambos ficam registrados.

def _ano_de_data_defesa(valor):
    """Extrai o ANO (int) de uma data tipo '23/02/2022 10:00h' ou '11/06/2021'."""
    if pd.isna(valor):
        return pd.NA
    texto = str(valor).strip()
    if not texto:
        return pd.NA
    data = texto.split()[0]              # descarta a hora, se houver
    partes = data.split('/')
    if len(partes) == 3 and partes[2].isdigit():
        return int(partes[2])
    return pd.NA


def _normalizar_md(valor):
    """Normaliza o rótulo de nível para 'Mestrado'/'Doutorado' (ou None)."""
    n = normalizar_texto(valor)
    if 'DOUTORADO' in n:
        return 'Doutorado'
    if 'MESTRADO' in n:
        return 'Mestrado'
    return None


def construir_indice_variacoes(df_alunos_local):
    """Índice reverso: variação de nome normalizada -> id_lattes do aluno.

    Uma variação usada por mais de um aluno é ambígua e é descartada do
    índice -- sem decisão fuzzy/probabilística; a linha fica de fora de
    df_aluno_titulos até revisão manual, em vez de resolver errado."""
    indice = {}
    for _, linha in df_alunos_local.iterrows():
        variacoes = {v.strip() for v in str(linha.get('variacoes_coautoria', '')).split('|') if v.strip()}
        for variacao in variacoes:
            indice.setdefault(variacao, set()).add(linha['id_lattes'])
    return {variacao: next(iter(ids)) for variacao, ids in indice.items() if len(ids) == 1}


print("Construindo índice de variações de nome dos alunos...")
indice_variacoes_alunos = construir_indice_variacoes(df_alunos) if not df_alunos.empty else {}

# Alias manual para orientando cujo nome, digitado pelo próprio professor no
# Lattes dele, não bate com nenhuma variação/permutação conhecida do aluno --
# typo do professor ou aluno que mudou de nome depois da orientação já ter
# sido registrada. A chave de resolução final continua sendo sempre o
# id_lattes do aluno (nunca um novo casamento por nome); arquivo opcional,
# mesmo padrão do alias_orientadores.csv.
df_alias_orientandos = pd.DataFrame(columns=['nome_no_lattes_professor', 'id_lattes_aluno'])
if Path(CAMINHO_ALIAS_ORIENTANDOS).exists():
    df_alias_orientandos = pd.read_csv(CAMINHO_ALIAS_ORIENTANDOS, dtype=str).dropna(
        subset=['nome_no_lattes_professor', 'id_lattes_aluno']
    )
mapa_alias_orientandos = dict(zip(
    df_alias_orientandos['nome_no_lattes_professor'].map(normalizar_texto),
    df_alias_orientandos['id_lattes_aluno'],
))


def resolver_id_aluno(nome):
    """Resolve um nome de aluno (texto livre) para o id_lattes do aluno --
    primeiro pelo índice de variações/permutações, depois pelo alias manual
    (dados_brutos/alias_orientandos.csv). Sempre uma correspondência exata
    (nunca fuzzy); o id_lattes do aluno é a chave primária em ambos os casos."""
    nome_norm = normalizar_texto(nome)
    return indice_variacoes_alunos.get(nome_norm) or mapa_alias_orientandos.get(nome_norm)


registros_titulos = []

# --- Fonte 1: CSV de defesas do PESC (sem professor associado) ---
df_defesas = pd.read_csv(CAMINHO_CSV_DEFESAS)
for _, linha in df_defesas.iterrows():
    nivel = _normalizar_md(linha.get('M/D'))
    id_lattes_aluno = resolver_id_aluno(linha.get('Nome do Autor'))
    if nivel and id_lattes_aluno:
        registros_titulos.append({
            'id_lattes_aluno': id_lattes_aluno,
            'nivel': nivel,
            'ano': _ano_de_data_defesa(linha.get('Data da Defesa')),
            'status': 'Concluída',
            'id_lattes_professor': pd.NA,
            'fonte': 'csv_defesas',
        })

# --- Fonte 2: df_orientacoes, TODOS os professores (apenas Mestrado/Doutorado) ---
if not df_orientacoes.empty:
    mask_niveis = df_orientacoes['nivel'].isin(['Mestrado', 'Doutorado'])
    for _, linha in df_orientacoes[mask_niveis].iterrows():
        id_lattes_aluno = resolver_id_aluno(linha.get('orientando'))
        if not id_lattes_aluno:
            continue
        ano = linha.get('ano_conclusao')
        registros_titulos.append({
            'id_lattes_aluno': id_lattes_aluno,
            'nivel': linha.get('nivel'),
            'ano': int(ano) if pd.notna(ano) else pd.NA,
            'status': linha.get('status'),
            'id_lattes_professor': linha.get('id_lattes'),
            'fonte': 'lattes_orientacoes',
        })

df_aluno_titulos = pd.DataFrame(
    registros_titulos,
    columns=['id_lattes_aluno', 'nivel', 'ano', 'status', 'id_lattes_professor', 'fonte'],
)
if not df_aluno_titulos.empty:
    df_aluno_titulos['ano'] = df_aluno_titulos['ano'].astype('Int64')
    df_aluno_titulos = df_aluno_titulos.drop_duplicates(
        subset=['id_lattes_aluno', 'nivel', 'id_lattes_professor', 'ano', 'fonte']
    ).reset_index(drop=True)

print(f"Total de registros de título (histórico completo, todas as fontes): {len(df_aluno_titulos)}")
alunos_distintos = df_aluno_titulos['id_lattes_aluno'].nunique() if not df_aluno_titulos.empty else 0
print(f"Alunos distintos com pelo menos um título: {alunos_distintos}")
display(df_aluno_titulos.head(10))


### 11.3 Verificação de coautoria nas duas tabelas unificadas

In [32]:
def montar_lista_variacoes(df_alunos_local):
    """Transforma a coluna 'variacoes_coautoria' de cada aluno em um conjunto de strings, agrupados por aluno."""
    variacoes = []
    if df_alunos_local.empty:
        return variacoes

    for _, linha in df_alunos_local.iterrows():
        nomes = [item.strip() for item in str(linha.get('variacoes_coautoria', '')).split('|') if item.strip()]
        if nomes:
            variacoes.append(set(nomes))

    return variacoes


def tem_coautoria_aluno(autores, variacoes_alunos):
    """Verifica se a string de autores de uma publicação contém alguma variação de nome de algum aluno.

    A comparação usa espaços como delimitadores nas duas pontas para evitar
    que uma variação curta (ex.: uma única inicial) seja encontrada como
    substring de outra palavra sem relação nenhuma.
    """
    if pd.isna(autores) or not str(autores).strip() or not variacoes_alunos:
        return False

    autores_normalizados = f" {normalizar_texto(autores)} "

    for variacoes in variacoes_alunos:
        for nome_normalizado in variacoes:
            if f" {nome_normalizado} " in autores_normalizados:
                return True

    return False


print("Marcando coautoria de alunos em df_periodicos_unificado e df_congressos_unificado...")
variacoes_alunos = montar_lista_variacoes(df_alunos)

for df_prod in [df_periodicos_unificado, df_congressos_unificado]:
    if 'autores' in df_prod.columns:
        df_prod['coautoria_aluno'] = df_prod['autores'].apply(
            lambda autores: tem_coautoria_aluno(autores, variacoes_alunos)
        )
    else:
        df_prod['coautoria_aluno'] = False

print(f"Periódicos com coautoria de aluno:   {int(df_periodicos_unificado['coautoria_aluno'].sum())}")
print(f"Conferências com coautoria de aluno: {int(df_congressos_unificado['coautoria_aluno'].sum())}")

Marcando coautoria de alunos em df_periodicos_unificado e df_congressos_unificado...


Periódicos com coautoria de aluno:   720
Conferências com coautoria de aluno: 1782


## 12. Propagação do Tratamento de Volta para as Tabelas por Fonte

As Seções 9-11 calcularam o enriquecimento (percentil, estrato, match,
coautoria de aluno) **uma única vez**, sobre a base já deduplicada. Esta
seção devolve esse resultado para `df_periodicos_bruto`/`df_congressos_bruto`
(que ainda têm uma linha por publicação **por fonte**), usando `chave_dedup`
como elo — o mesmo papel que uma FK exerceria, só que resolvido em pandas
antes do `INSERT`, o que evita ter que rodar `UPDATE`s no DuckDB depois.

Junto com o enriquecimento vão também duas colunas que existem justamente
para viabilizar os relatórios de lacuna:

- **`chave_dedup`** — o identificador da publicação dentro da lista daquele
  professor, idêntico nas 7 tabelas. É por ele que se cruza a tabela
  unificada com qualquer tabela por fonte.
- **`fontes`** — a lista das bases em que aquela publicação foi encontrada
  (ex.: `LATTES,SCOPUS`), já refletindo o resultado da deduplicação.

O resultado é dividido de volta nas três fontes — essas seis variáveis
(`df_artigos_periodico_lattes/orcid/scopus`,
`df_artigos_congresso_lattes/orcid/scopus`) populam as 6 tabelas por fonte
na Seção 13. `montar_relatorio_lacunas` abre a coluna `fontes` em três
booleanos (`em_lattes`/`em_orcid`/`em_scopus`) para conferência rápida aqui
no notebook; no banco, a consulta equivalente é:

```sql
-- artigos de periódico que o professor tem, mas que faltam no ORCID
SELECT u.id_lattes, u.titulo_artigo, u.doi, u.fontes
FROM tb_artigo_periodico u
LEFT JOIN tb_artigo_periodico_orcid o ON o.chave_dedup = u.chave_dedup
WHERE o.chave_dedup IS NULL;
```

In [33]:
COLUNAS_ENRIQUECIMENTO_PERIODICO = [
    'match_adequado', 'id_scopus', 'titulo_revista_scopus', 'maior_percentil',
    'codigo_area_maior_percentil', 'area_maior_percentil', 'issn',
    'computation_area', 'coautoria_aluno',
]

COLUNAS_ENRIQUECIMENTO_CONGRESSO = [
    'sigla_evento_google', 'titulo_evento_google', 'estrato', 'tipo_match', 'coautoria_aluno',
]

# Além do enriquecimento, cada linha por fonte recebe também `fontes` (todas
# as bases em que aquela publicação foi encontrada) e `chave_dedup` (o elo
# com a tabela unificada). São essas duas colunas que permitem responder
# "quais artigos do professor X estão no Lattes mas faltam no ORCID?".
COLUNAS_PERIODICO_POR_FONTE = COLUNAS_PERIODICO + ['fontes', 'chave_dedup']
COLUNAS_CONGRESSO_POR_FONTE = COLUNAS_CONGRESSO + ['fontes', 'chave_dedup']


def propagar_enriquecimento(df_bruto, df_unificado_tratado, colunas_enriquecimento):
    """Devolve para a base bruta (uma linha por publicação por fonte) os
    valores já tratados na base unificada, ligando as duas por `chave_dedup`.
    Além das colunas de cruzamento, traz `fontes`, para que cada linha por
    fonte saiba em quais outras bases aquela publicação também aparece."""
    colunas_a_trazer = colunas_enriquecimento + ['fontes']
    df_enriquecimento = df_unificado_tratado[['chave_dedup'] + colunas_a_trazer]
    return (
        df_bruto.drop(columns=colunas_a_trazer, errors='ignore')
        .merge(df_enriquecimento, on='chave_dedup', how='left')
    )


print("Propagando os valores tratados de volta para as linhas por fonte...")
df_periodicos_bruto = propagar_enriquecimento(df_periodicos_bruto, df_periodicos_unificado, COLUNAS_ENRIQUECIMENTO_PERIODICO)
df_congressos_bruto = propagar_enriquecimento(df_congressos_bruto, df_congressos_unificado, COLUNAS_ENRIQUECIMENTO_CONGRESSO)


def dividir_por_fonte(df_bruto, colunas_finais, fonte):
    """Isola as linhas de uma fonte específica, já no schema final."""
    return df_bruto[df_bruto['fonte'] == fonte][colunas_finais].reset_index(drop=True)


df_artigos_periodico_lattes = dividir_por_fonte(df_periodicos_bruto, COLUNAS_PERIODICO_POR_FONTE, 'LATTES')
df_artigos_periodico_orcid = dividir_por_fonte(df_periodicos_bruto, COLUNAS_PERIODICO_POR_FONTE, 'ORCID')
df_artigos_periodico_scopus = dividir_por_fonte(df_periodicos_bruto, COLUNAS_PERIODICO_POR_FONTE, 'SCOPUS')

df_artigos_congresso_lattes = dividir_por_fonte(df_congressos_bruto, COLUNAS_CONGRESSO_POR_FONTE, 'LATTES')
df_artigos_congresso_orcid = dividir_por_fonte(df_congressos_bruto, COLUNAS_CONGRESSO_POR_FONTE, 'ORCID')
df_artigos_congresso_scopus = dividir_por_fonte(df_congressos_bruto, COLUNAS_CONGRESSO_POR_FONTE, 'SCOPUS')


def montar_relatorio_lacunas(df_unificado, rotulo):
    """Monta a visão de lacunas: uma linha por publicação de cada professor,
    com uma coluna booleana por base indicando presença/ausência. É a mesma
    informação que a coluna `fontes` carrega, só que já aberta para conferência
    rápida aqui no notebook (no banco, a consulta equivalente cruza
    `tb_artigo_*` com `tb_artigo_*_<fonte>` por `chave_dedup`)."""
    if df_unificado.empty:
        return pd.DataFrame(columns=['id_lattes', 'titulo_artigo', 'chave_dedup',
                                     'em_lattes', 'em_orcid', 'em_scopus'])
    relatorio = df_unificado[['id_lattes', 'titulo_artigo', 'doi', 'chave_dedup', 'fontes']].copy()
    for fonte in ['LATTES', 'ORCID', 'SCOPUS']:
        relatorio[f'em_{fonte.lower()}'] = relatorio['fontes'].fillna('').str.split(',').apply(lambda lista: fonte in lista)
    faltando = (~relatorio[['em_lattes', 'em_orcid', 'em_scopus']]).sum()
    print(f"  {rotulo}: {len(relatorio)} publicações | ausentes no Lattes: {faltando['em_lattes']} | "
          f"no ORCID: {faltando['em_orcid']} | no Scopus: {faltando['em_scopus']}")
    return relatorio


print("\nPropagação concluída. Volumes finais por fonte:")
print(f"  Periódico  -- LATTES: {len(df_artigos_periodico_lattes)} | ORCID: {len(df_artigos_periodico_orcid)} | SCOPUS: {len(df_artigos_periodico_scopus)}")
print(f"  Congresso  -- LATTES: {len(df_artigos_congresso_lattes)} | ORCID: {len(df_artigos_congresso_orcid)} | SCOPUS: {len(df_artigos_congresso_scopus)}")
print(f"  Unificado (deduplicado) -- Periódico: {len(df_periodicos_unificado)} | Congresso: {len(df_congressos_unificado)}")

print("\nLacunas por base:")
df_lacunas_periodicos = montar_relatorio_lacunas(df_periodicos_unificado, 'Periódicos')
df_lacunas_congressos = montar_relatorio_lacunas(df_congressos_unificado, 'Congressos')
display(df_lacunas_periodicos.head())

Propagando os valores tratados de volta para as linhas por fonte...

Propagação concluída. Volumes finais por fonte:
  Periódico  -- LATTES: 1589 | ORCID: 682 | SCOPUS: 1196
  Congresso  -- LATTES: 3088 | ORCID: 719 | SCOPUS: 1463
  Unificado (deduplicado) -- Periódico: 1746 | Congresso: 3428

Lacunas por base:
  Periódicos: 1746 publicações | ausentes no Lattes: 157 | no ORCID: 1064 | no Scopus: 550
  Congressos: 3428 publicações | ausentes no Lattes: 340 | no ORCID: 2709 | no Scopus: 1965


,id_lattes,titulo_artigo,doi,chave_dedup,fontes,em_lattes,em_orcid,em_scopus
0,0211300683784278,On the (In)Dependence of the Peano Axioms for ...,http://dx.doi.org/10.1080/01445340.2021.1971005,0211300683784278|DOI:10.1080/01445340.2021.197...,"LATTES,ORCID,SCOPUS",True,True,True
1,0211300683784278,Short proofs on the structure of general parti...,http://dx.doi.org/10.1016/j.dam.2020.09.007,0211300683784278|DOI:10.1016/j.dam.2020.09.007,"LATTES,SCOPUS",True,False,True
2,0211300683784278,Transversals of longest paths,http://dx.doi.org/10.1016/j.disc.2019.111717,0211300683784278|DOI:10.1016/j.disc.2019.111717,"LATTES,SCOPUS",True,False,True
3,0211300683784278,Intersection of longest paths in graph classes,http://dx.doi.org/10.1016/j.dam.2019.03.022,0211300683784278|DOI:10.1016/j.dam.2019.03.022,"LATTES,SCOPUS",True,False,True
4,0211300683784278,"L(2,1)-labelling of graphs with few P4?s",10.1016/j.disopt.2016.01.006,0211300683784278|DOI:10.1016/j.disopt.2016.01.006,"LATTES,SCOPUS",True,False,True


## 13. Persistência Consolidada no DuckDB

Cria (se não existir) o schema relacional em `pesquisadores_teste.duckdb` e
carrega os DataFrames tratados. **Doze tabelas** ao todo:

- `tb_professores` — tabela "mãe", uma linha por professor (chave
  `id_lattes`); agora com `orcid_id`/`scopus_author_id` (nullable).
- `tb_alunos` — uma linha por aluno, com as variações de nome da Seção 11.
- `tb_aluno_titulos` — **nova**: histórico completo de títulos por aluno
  (1:N), uma linha por `(aluno, nível, professor, ano, fonte)`. Substitui as
  antigas colunas `tb_alunos.titulo`/`tb_alunos.data_defesa`, que colapsavam
  cada aluno a um único título (o mais alto).
- `tb_artigo_periodico` / `tb_artigo_conferencia` — **mesmo formato de
  antes** (é o que `app.py` consulta), agora com uma coluna `fontes`
  adicional; populadas a partir da base unificada e deduplicada.
- `tb_orientacoes` — uma orientação por linha.
- **Seis tabelas novas**, uma por fonte × tipo de produção
  (`tb_artigo_periodico_lattes/orcid/scopus`,
  `tb_artigo_conferencia_lattes/orcid/scopus`) — mesmo schema das tabelas
  unificadas (mais uma coluna `fonte` de valor único), preservando uma
  linha por publicação por fonte, já com os campos de cruzamento
  propagados na Seção 12.

Todas as tabelas de artigos têm `FOREIGN KEY (id_lattes) REFERENCES
tb_professores(id_lattes)` — `id_lattes` continua sendo a única chave
primária/estrangeira do modelo; `orcid_id` e `scopus_author_id` vivem como
atributos de `tb_professores`, não como chaves estrangeiras separadas.
`tb_aluno_titulos` referencia `tb_alunos(id_lattes)` e, opcionalmente (pode
ser nulo, fonte `csv_defesas`), `tb_professores(id_lattes)`.

A carga é feita em modo *replace*: as tabelas filhas são limpas antes da
tabela mãe (para não violar a integridade referencial) e, em seguida, todo o
conteúdo tratado em memória é inserido novamente.


### 13.1 Criação do schema (tabelas, sequências e chaves estrangeiras)

In [ ]:
print("Conectando ao DuckDB e criando o schema (se ainda não existir)...")
con = duckdb.connect(ARQUIVO_DUCKDB_DESTINO)

# --- Tabela mãe: Professores ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_professores (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    sexo VARCHAR,
    rotulo VARCHAR,
    periodo VARCHAR,
    bolsa_produtividade VARCHAR,
    endereco_profissional VARCHAR,
    atualizacao_cv TIMESTAMP,
    url VARCHAR,
    texto_resumo VARCHAR,
    orcid_id VARCHAR,
    scopus_author_id VARCHAR,
    data_ingresso INTEGER
);
""")

# --- Tabela de Alunos ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_alunos (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    nome_completo_normalizado VARCHAR,
    nome_citacoes_normalizadas VARCHAR,
    variacoes_coautoria VARCHAR
);
""")

# --- Sequências para os IDs automáticos das tabelas filhas ---
for nome_sequencia in [
    'seq_id_artigo_periodico', 'seq_id_artigo_conferencia', 'seq_id_orientacao',
    'seq_id_artigo_periodico_lattes', 'seq_id_artigo_periodico_orcid', 'seq_id_artigo_periodico_scopus',
    'seq_id_artigo_conferencia_lattes', 'seq_id_artigo_conferencia_orcid', 'seq_id_artigo_conferencia_scopus',
    'seq_id_aluno_titulo', 'seq_id_doi_descartado',
]:
    con.execute(f"CREATE SEQUENCE IF NOT EXISTS {nome_sequencia};")

# --- Tabela Filha: Artigos de Periódico (unificada, cruzada com Scopus) ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_artigo_periodico (
    id_artigo_periodico INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_periodico'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    match_adequado BOOLEAN,
    coautoria_aluno BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    fontes VARCHAR,
    chave_dedup VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha: Artigos de Conferência (unificada, cruzada com a base de eventos) ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_artigo_conferencia (
    id_artigo_conferencia INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_conferencia'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    coautoria_aluno BOOLEAN,
    fontes VARCHAR,
    chave_dedup VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha: Orientações ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_orientacoes (
    id_orientacao INTEGER PRIMARY KEY DEFAULT nextval('seq_id_orientacao'),
    id_lattes VARCHAR,
    titulo_trabalho VARCHAR,
    ano_inicio INTEGER,
    orientando VARCHAR,
    tipo_trabalho VARCHAR,
    instituicao VARCHAR,
    curso VARCHAR,
    status VARCHAR,
    nivel VARCHAR,
    ano_conclusao INTEGER,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha: Histórico completo de títulos dos alunos (1:N) ---
# Substitui as antigas colunas tb_alunos.titulo/tb_alunos.data_defesa, que
# colapsavam cada aluno a um único título (o mais alto). Aqui um mesmo aluno
# pode ter várias linhas (ex.: Mestrado com um professor, Doutorado com
# outro). id_lattes_professor é nulo quando a linha vem só da CSV de
# defesas do PESC (fonte='csv_defesas'), que não registra orientador.
con.execute("""
CREATE TABLE IF NOT EXISTS tb_aluno_titulos (
    id_titulo INTEGER PRIMARY KEY DEFAULT nextval('seq_id_aluno_titulo'),
    id_lattes_aluno VARCHAR NOT NULL,
    nivel VARCHAR,
    ano INTEGER,
    status VARCHAR,
    id_lattes_professor VARCHAR,
    fonte VARCHAR,
    FOREIGN KEY (id_lattes_aluno) REFERENCES tb_alunos(id_lattes),
    FOREIGN KEY (id_lattes_professor) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha: DOIs descartados no saneamento (Seção 8.2.1) ---
# Não é dado de produção: é o registro do que o pipeline recusou usar como DOI,
# e serve para o docente localizar e corrigir a entrada no próprio currículo.
# `doi_descartado` guarda o valor exatamente como veio do Lattes -- é por ele
# que a linha é encontrada no CV.
con.execute("""
CREATE TABLE IF NOT EXISTS tb_dois_descartados (
    id_descarte INTEGER PRIMARY KEY DEFAULT nextval('seq_id_doi_descartado'),
    id_lattes VARCHAR,
    tipo VARCHAR,
    fonte VARCHAR,
    titulo_artigo VARCHAR,
    ano INTEGER,
    doi_descartado VARCHAR,
    motivo VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")


# --- Seis tabelas novas: uma por fonte x tipo de produção (dado pré-deduplicação) ---
COLUNAS_SQL_ARTIGO_PERIODICO_FONTE = """
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    match_adequado BOOLEAN,
    coautoria_aluno BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    citacoes_scopus INTEGER,
    fonte VARCHAR,
    fontes VARCHAR,
    chave_dedup VARCHAR,
"""

COLUNAS_SQL_ARTIGO_CONFERENCIA_FONTE = """
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    coautoria_aluno BOOLEAN,
    citacoes_scopus INTEGER,
    fonte VARCHAR,
    fontes VARCHAR,
    chave_dedup VARCHAR,
"""


def criar_tabela_por_fonte(nome_tabela, nome_sequencia, nome_pk, colunas_sql):
    con.execute(f"""
        CREATE TABLE IF NOT EXISTS {nome_tabela} (
            {nome_pk} INTEGER PRIMARY KEY DEFAULT nextval('{nome_sequencia}'),
            {colunas_sql}
            FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
        );
    """)


for fonte_sufixo in ['lattes', 'orcid', 'scopus']:
    criar_tabela_por_fonte(
        f'tb_artigo_periodico_{fonte_sufixo}', f'seq_id_artigo_periodico_{fonte_sufixo}',
        f'id_artigo_periodico_{fonte_sufixo}', COLUNAS_SQL_ARTIGO_PERIODICO_FONTE,
    )
    criar_tabela_por_fonte(
        f'tb_artigo_conferencia_{fonte_sufixo}', f'seq_id_artigo_conferencia_{fonte_sufixo}',
        f'id_artigo_conferencia_{fonte_sufixo}', COLUNAS_SQL_ARTIGO_CONFERENCIA_FONTE,
    )

# Garante que bancos criados em uma versão anterior do schema (sem estas
# colunas) sejam atualizados ao reexecutar o notebook. Cada ALTER é
# protegido por try/except porque o DuckDB ainda não suporta
# "ADD COLUMN IF NOT EXISTS" de forma totalmente idempotente em todas as versões.
for alter_sql in [
    "ALTER TABLE tb_artigo_periodico ADD COLUMN autores VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN doi VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN fontes VARCHAR",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN fontes VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN chave_dedup VARCHAR",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN chave_dedup VARCHAR",
] + [
    f"ALTER TABLE tb_artigo_{tipo}_{fonte} ADD COLUMN {coluna} VARCHAR"
    for tipo in ['periodico', 'conferencia']
    for fonte in ['lattes', 'orcid', 'scopus']
    for coluna in ['fontes', 'chave_dedup']
] + [
    # Citacoes por publicacao (Secao 7). So a fonte Scopus preenche, mas a coluna
    # existe nas seis tabelas para que as tres fontes continuem com o mesmo
    # schema -- um UNION ALL entre elas nao pode depender de qual fonte e'.
    f"ALTER TABLE tb_artigo_{tipo}_{fonte} ADD COLUMN citacoes_scopus INTEGER"
    for tipo in ['periodico', 'conferencia']
    for fonte in ['lattes', 'orcid', 'scopus']
] + [
    "ALTER TABLE tb_alunos ADD COLUMN nome_completo_normalizado VARCHAR",
    "ALTER TABLE tb_alunos ADD COLUMN nome_citacoes_normalizadas VARCHAR",
    "ALTER TABLE tb_alunos ADD COLUMN variacoes_coautoria VARCHAR",
    "ALTER TABLE tb_professores ADD COLUMN orcid_id VARCHAR",
    "ALTER TABLE tb_professores ADD COLUMN scopus_author_id VARCHAR",
    "ALTER TABLE tb_professores ADD COLUMN data_ingresso INTEGER",
]:
    try:
        con.execute(alter_sql)
    except Exception:
        pass  # Coluna já existe — nada a fazer

# Retira as antigas titulo/data_defesa de tb_alunos (substituídas por
# tb_aluno_titulos) em bancos criados por uma versão anterior do schema.
for drop_sql in [
    "ALTER TABLE tb_alunos DROP COLUMN titulo",
    "ALTER TABLE tb_alunos DROP COLUMN data_defesa",
]:
    try:
        con.execute(drop_sql)
    except Exception:
        pass  # Coluna já não existe — nada a fazer

print("Schema pronto: 6 tabelas principais (incl. tb_aluno_titulos) + 6 tabelas por fonte "
      "+ tb_dois_descartados (criadas ou já existentes).")


### 13.2 Carga dos dados (limpeza das tabelas antigas + inserção)

In [ ]:
print("Limpando dados antigos antes da nova carga (filhas primeiro, mãe depois)...")
# A ordem importa: as tabelas filhas têm FOREIGN KEY para tb_professores
# e/ou tb_alunos, então precisam ser esvaziadas antes das tabelas mãe.
tabelas_filhas = [
    'tb_artigo_periodico', 'tb_artigo_conferencia', 'tb_orientacoes', 'tb_aluno_titulos',
    'tb_artigo_periodico_lattes', 'tb_artigo_periodico_orcid', 'tb_artigo_periodico_scopus',
    'tb_artigo_conferencia_lattes', 'tb_artigo_conferencia_orcid', 'tb_artigo_conferencia_scopus',
    'tb_dois_descartados',
]
for tabela in tabelas_filhas:
    con.execute(f"DELETE FROM {tabela}")
con.execute("DELETE FROM tb_alunos")
con.execute("DELETE FROM tb_professores")

print("Inserindo os dados tratados...")

# --- Tabela Mãe: Professores ---
if not df_pessoas.empty:
    con.execute("""
        INSERT INTO tb_professores (
            id_lattes, nome_completo, nome_citacoes, sexo, rotulo, periodo,
            bolsa_produtividade, endereco_profissional, atualizacao_cv, url,
            texto_resumo, orcid_id, scopus_author_id, data_ingresso
        )
        SELECT
            id_lattes, nome_completo, nome_citacoes, sexo, rotulo, periodo,
            bolsa_produtividade, endereco_profissional, atualizacao_cv, url,
            texto_resumo, orcid_id, scopus_author_id, data_ingresso
        FROM df_pessoas
    """)

# --- Tabela de Alunos ---
if not df_alunos.empty:
    con.execute("""
        INSERT INTO tb_alunos (
            id_lattes, nome_completo, nome_citacoes,
            nome_completo_normalizado, nome_citacoes_normalizadas, variacoes_coautoria
        )
        SELECT
            id_lattes, nome_completo, nome_citacoes,
            nome_completo_normalizado, nome_citacoes_normalizadas, variacoes_coautoria
        FROM df_alunos
    """)

# --- Tabela Filha: Histórico completo de títulos dos alunos ---
if not df_aluno_titulos.empty:
    con.execute("""
        INSERT INTO tb_aluno_titulos (
            id_lattes_aluno, nivel, ano, status, id_lattes_professor, fonte
        )
        SELECT
            id_lattes_aluno, nivel, ano, status, id_lattes_professor, fonte
        FROM df_aluno_titulos
    """)

# --- Tabela Unificada: Artigos de Periódico ---
if not df_periodicos_unificado.empty:
    con.execute("""
        INSERT INTO tb_artigo_periodico (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area, fontes, chave_dedup
        FROM df_periodicos_unificado
    """)

# --- Tabela Unificada: Artigos de Conferência ---
if not df_congressos_unificado.empty:
    con.execute("""
        INSERT INTO tb_artigo_conferencia (
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno, fontes, chave_dedup
        FROM df_congressos_unificado
    """)

# --- Tabela Filha: Orientações ---
if not df_orientacoes.empty:
    con.execute("""
        INSERT INTO tb_orientacoes (
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        )
        SELECT
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        FROM df_orientacoes
    """)

# --- Seis tabelas por fonte (dado pré-deduplicação, já com o tratamento propagado) ---
mapa_tabelas_periodico_fonte = {
    'tb_artigo_periodico_lattes': df_artigos_periodico_lattes,
    'tb_artigo_periodico_orcid': df_artigos_periodico_orcid,
    'tb_artigo_periodico_scopus': df_artigos_periodico_scopus,
}
for nome_tabela, df_fonte in mapa_tabelas_periodico_fonte.items():
    if df_fonte.empty:
        continue
    con.register('df_fonte_periodico_tmp', df_fonte)
    con.execute(f"""
        INSERT INTO {nome_tabela} (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area, citacoes_scopus,
            fonte, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area,
            TRY_CAST(citacoes_scopus AS INTEGER), fonte, fontes, chave_dedup
        FROM df_fonte_periodico_tmp
    """)
    con.unregister('df_fonte_periodico_tmp')

mapa_tabelas_congresso_fonte = {
    'tb_artigo_conferencia_lattes': df_artigos_congresso_lattes,
    'tb_artigo_conferencia_orcid': df_artigos_congresso_orcid,
    'tb_artigo_conferencia_scopus': df_artigos_congresso_scopus,
}
for nome_tabela, df_fonte in mapa_tabelas_congresso_fonte.items():
    if df_fonte.empty:
        continue
    con.register('df_fonte_congresso_tmp', df_fonte)
    con.execute(f"""
        INSERT INTO {nome_tabela} (
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno,
            citacoes_scopus, fonte, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno,
            TRY_CAST(citacoes_scopus AS INTEGER), fonte, fontes, chave_dedup
        FROM df_fonte_congresso_tmp
    """)
    con.unregister('df_fonte_congresso_tmp')

# --- DOIs descartados no saneamento ---
if not df_dois_descartados.empty:
    con.register('df_dois_descartados_tmp', df_dois_descartados)
    con.execute("""
        INSERT INTO tb_dois_descartados (
            id_lattes, tipo, fonte, titulo_artigo, ano, doi_descartado, motivo
        )
        SELECT
            id_lattes, tipo, fonte, titulo_artigo,
            TRY_CAST(ano AS INTEGER), doi_descartado, motivo
        FROM df_dois_descartados_tmp
    """)
    con.unregister('df_dois_descartados_tmp')


con.close()

print(f"Processo finalizado! Banco '{ARQUIVO_DUCKDB_DESTINO}' atualizado com o schema completo (13 tabelas).")


## 14. Situação dos Orientandos do Programa no Lattes do Orientador

Complementa a Seção 11: cruza `dados_brutos/lista_alunos_pesc.xlsx` (registro
administrativo oficial do programa — aluno, nível, ano de ingresso e
orientador) com `df_aluno_titulos` (fonte `lattes_orientacoes` — o histórico
completo de títulos, já resolvido aluno↔professor na Seção 11.2.3) para
descobrir quais alunos do programa ainda não foram incluídos no Lattes do seu
orientador. O resultado alimenta o relatório "Alunos do programa faltando no
Lattes do orientador" em `app.py`.

Diferente da versão anterior deste notebook, esta seção **não refaz** o match
aluno↔orientando: ela só verifica se o par `(id_lattes_aluno, nivel,
id_lattes_professor)` já existe em `df_aluno_titulos` — o cruzamento em si
(variações de nome, normalização) é feito uma única vez na Seção 11.2.3 e
reaproveitado aqui e pelo relatório "Títulos dos alunos no Lattes de cada
professor".

Seção inteiramente aditiva: não lê nem modifica nenhuma variável usada pela
Seção 13 (persistência das doze tabelas de sempre) e persiste numa tabela
nova (`tb_situacao_orientandos`), com sua própria conexão DuckDB.

### Estratégia de casamento (sempre exata, nunca fuzzy)

1. **Aluno → `id_lattes`**: extraído diretamente da URL na coluna `Lattes`
   do xlsx (bate 100% com os arquivos em `dados_brutos/alunos/` nos dados
   atuais).
2. **Orientador (texto livre, às vezes dois nomes separados por `/` em caso
   de coorientação) → `id_lattes` do professor**: reaproveita
   `resolver_id_professor`, definido uma única vez na Seção 11.2.1 (mesma
   função usada para resolver o orientador declarado pelo próprio aluno na
   Seção 11.2.2) -- comparação exata por conjunto de tokens do nome
   (maiúsculas, sem acento, ignorando "de/da/do/dos/das/e"), exigindo que os
   tokens do orientador sejam subconjunto dos tokens do nome completo do
   professor, resolvido apenas quando há exatamente **um** professor
   candidato. Nomes não resolvidos automaticamente (erro de digitação,
   abreviação, coorientador externo) podem ser corrigidos manualmente em
   `dados_brutos/alias_orientadores.csv` (colunas `nome_no_xlsx,id_lattes`)
   -- mesmo padrão do `lista_pessoas.csv`, opcional, e sem afetar mais nada
   no pipeline.
3. **Aluno × professor × nível já registrado no Lattes**: consulta direta em
   `df_aluno_titulos` (fonte `lattes_orientacoes`) pelo trio
   `(id_lattes_aluno, nivel, id_lattes_professor)` -- o mesmo aluno pode
   aparecer no Lattes para o Mestrado mas ainda não para o Doutorado, por
   exemplo, pois cada nível é uma linha independente.

Cada (aluno, orientador, nível) vira uma linha em `tb_situacao_orientandos`,
com `encontrado_no_lattes` indicando se aquele vínculo já está registrado no
Lattes do professor.

### Terceira fonte: o que o próprio aluno declara no Lattes dele

Além da planilha administrativa e do Lattes do professor, a Seção 11.2.2 já
extraiu `df_formacao_aluno` -- o que o ALUNO declarou como seu orientador em
`formacao_academica`, por nível. Esta seção agora também cruza essa terceira
fonte, sempre por `id_lattes` (nunca por nome), e classifica cada linha em
`nivel_confianca`:

- **`confirmado`**: o vínculo está tanto no Lattes do professor quanto na
  formação acadêmica do próprio aluno -- duas fontes independentes batendo.
- **`confirmado_parcial`**: só uma das duas fontes (professor OU aluno)
  confirma o vínculo.
- **`sem_confirmacao`**: nem o professor nem o aluno confirmam -- só a
  planilha administrativa registra essa relação.
- **`divergente`**: o aluno declarou, para aquele nível, um orientador que a
  planilha administrativa nem sequer lista como (co)orientador -- sinal de
  que a planilha, o Lattes do professor ou o do aluno estão desatualizados
  entre si. Nunca decidido sozinho, sempre sinalizado para revisão manual.

As colunas `titulo_trabalho_aluno` e `ano_obtencao_aluno` (quando o aluno
confirma aquele professor especificamente) vêm dessa mesma fonte.


In [36]:
print("Carregando lista administrativa de alunos do programa...")
df_lista_alunos_pesc = pd.read_excel(CAMINHO_XLSX_ALUNOS_PESC)
df_lista_alunos_pesc['id_lattes_aluno'] = (
    df_lista_alunos_pesc['Lattes'].astype(str).str.extract(r'(\d+)')[0].str.strip()
)

# Explode coorientação ('Nome A/Nome B' -> uma linha por orientador), já que
# o relatório é por professor: cada coorientador precisa ver o aluno na sua
# própria lista de pendências.
df_situacao_bruto = df_lista_alunos_pesc.assign(
    orientador_texto=df_lista_alunos_pesc['Orientador'].astype(str).str.split('/')
).explode('orientador_texto')
df_situacao_bruto['orientador_texto'] = df_situacao_bruto['orientador_texto'].str.strip()
df_situacao_bruto = df_situacao_bruto[df_situacao_bruto['orientador_texto'] != '']

# Colapsa duplicatas administrativas do mesmo (aluno, nível, orientador) --
# a planilha tem alguns poucos casos de linha repetida para o mesmo grau --
# mantendo o registro de ano mais recente.
df_situacao_bruto = (
    df_situacao_bruto.sort_values('Ano')
    .drop_duplicates(subset=['id_lattes_aluno', 'M/D', 'orientador_texto'], keep='last')
    .reset_index(drop=True)
)

print(f"Linhas administrativas (após explodir coorientação e colapsar duplicatas): {len(df_situacao_bruto)}")
display(df_situacao_bruto[['Aluno', 'M/D', 'Ano', 'orientador_texto']].head())

Carregando lista administrativa de alunos do programa...
Linhas administrativas (após explodir coorientação e colapsar duplicatas): 365


,Aluno,M/D,Ano,orientador_texto
0,Alan de Oliveira Lyra,Doutorado,2021,Jano Moreira de Souza
1,Antonio Lacerda Junior,Doutorado,2021,Franklin Marquezino
2,Antonio Lacerda Junior,Doutorado,2021,Marcia Cerioli
3,Guina Guadalupe Sotomayor Alzamora,Doutorado,2021,Marcia Fampa
4,Herbert Salazar dos Santos,Doutorado,2021,Jano Moreira de Souza


In [37]:
print("Resolvendo nomes de orientador para id_lattes de professor (reaproveitando resolver_id_professor da Seção 11.2.1)...")
df_situacao_bruto['id_lattes_professor'] = df_situacao_bruto['orientador_texto'].apply(resolver_id_professor)

df_orientador_nao_localizado = (
    df_situacao_bruto[df_situacao_bruto['id_lattes_professor'].isna()]
    [['Aluno', 'orientador_texto']]
    .drop_duplicates()
    .sort_values('orientador_texto')
)
if df_orientador_nao_localizado.empty:
    print("OK: todos os nomes de orientador foram resolvidos.")
else:
    print(
        f"AVISO: {len(df_orientador_nao_localizado)} nome(s) de orientador não localizado(s) "
        f"em tb_professores -- adicione um alias em '{CAMINHO_ALIAS_ORIENTADORES}' "
        "(colunas nome_no_xlsx,id_lattes) se algum for um typo/abreviação de professor existente. "
        "Essas linhas ficam de fora de tb_situacao_orientandos até serem resolvidas."
    )
    display(df_orientador_nao_localizado)

Resolvendo nomes de orientador para id_lattes de professor...
AVISO: 34 nome(s) de orientador não localizado(s) em tb_professores -- adicione um alias em 'dados_brutos/alias_orientadores.csv' (colunas nome_no_xlsx,id_lattes) se algum for um typo/abreviação de professor existente. Essas linhas ficam de fora de tb_situacao_orientandos até serem resolvidas.


,Aluno,orientador_texto
79,Amanda Matos Ferreira,Abilio
23,Felipe Schreiber Fernandes,Abilio Lucena
56,Vitor Mazal Krauss,Abilio Lucena
58,Wagner Lima Monteiro,Abilio Lucena
266,André Luis Alves Martins,Abilio Lucena
70,Matheus Abreu da Costa Corrêa,Abilio Pereira de Lucena Filho
107,Juliana Nunes Rangel,Abilio Pereira de Lucena Filho
111,Matheus Degliomini Silva,Abilio Pereira de Lucena Filho
127,Alexandre Almeida de Oliveira,Abilio Pereira de Lucena Filho
117,Rodrigo Coacci,Claudio Amorim


In [ ]:
print("Casando cada aluno com o Lattes do orientador resolvido (reaproveitando df_aluno_titulos da Seção 11.2.3)...")

# Conjunto de trios já confirmados no Lattes de algum professor (fonte
# 'lattes_orientacoes' -- exclui 'csv_defesas', que não tem professor).
# Reaproveita o cruzamento exato feito uma única vez na Seção 11.2.3, em vez
# de refazer o match aluno x orientando aqui.
pares_lattes_orientacoes = set()
if not df_aluno_titulos.empty:
    df_lattes_titulos = df_aluno_titulos[df_aluno_titulos['fonte'] == 'lattes_orientacoes']
    pares_lattes_orientacoes = set(
        zip(
            df_lattes_titulos['id_lattes_aluno'],
            df_lattes_titulos['nivel'],
            df_lattes_titulos['id_lattes_professor'],
        )
    )


def aluno_encontrado_no_lattes(id_lattes_aluno, id_lattes_professor, nivel):
    if not id_lattes_professor:
        return False
    return (id_lattes_aluno, nivel, id_lattes_professor) in pares_lattes_orientacoes


# --- Terceira fonte: formação acadêmica declarada pelo próprio aluno (Seção 11.2.2) ---

# Todos os professores que o aluno já declarou como orientador, por (aluno,
# nível) -- pode haver mais de um registro para o mesmo nível (ex.: mestrado
# interrompido em outra instituição antes de entrar no PESC), então isto é
# um conjunto, não um valor único.
professores_declarados_pelo_aluno = (
    df_formacao_aluno[df_formacao_aluno['id_lattes_professor_via_aluno'].notna()]
    .groupby(['id_lattes_aluno', 'nivel'])['id_lattes_professor_via_aluno']
    .apply(set)
    .to_dict()
    if not df_formacao_aluno.empty else {}
)

# Todos os (co)orientadores que a PLANILHA ADMINISTRATIVA lista, por (aluno,
# nível) -- usado só para decidir 'divergente': o aluno declarou um professor
# que a planilha nem lista como (co)orientador daquele nível?
coorientadores_por_aluno_nivel = (
    df_situacao_bruto[df_situacao_bruto['id_lattes_professor'].notna()]
    .groupby(['id_lattes_aluno', 'M/D'])['id_lattes_professor']
    .apply(set)
    .to_dict()
)


def dados_formacao_aluno(id_lattes_aluno, nivel, id_lattes_professor):
    """Procura, entre os registros de formação acadêmica que o próprio aluno
    declarou para aquele nível, um cujo orientador resolvido bata com ESTE
    professor específico. Retorna (titulo, ano_obtencao, confirmado)."""
    if df_formacao_aluno.empty:
        return None, None, False

    linhas = df_formacao_aluno[
        (df_formacao_aluno['id_lattes_aluno'] == id_lattes_aluno)
        & (df_formacao_aluno['nivel'] == nivel)
        & (df_formacao_aluno['id_lattes_professor_via_aluno'] == id_lattes_professor)
    ]
    if linhas.empty:
        return None, None, False

    linha = linhas.iloc[0]
    ano_obtencao = pd.to_numeric(linha['ano_conclusao'], errors='coerce')
    ano_obtencao = int(ano_obtencao) if pd.notna(ano_obtencao) else None
    return linha['titulo_trabalho_aluno'], ano_obtencao, True


def nivel_de_confianca(id_lattes_aluno, nivel, encontrado_no_lattes, confirmado_pelo_aluno):
    """Sempre exata (nunca fuzzy): triangula as 3 fontes por id_lattes.

    - 'divergente': o aluno declarou, para este nível, um professor que a
      planilha administrativa nem lista como (co)orientador -- sinal de
      desatualização entre as fontes, para revisão manual.
    - 'confirmado': Lattes do professor E formação do aluno batem.
    - 'confirmado_parcial': só uma das duas fontes extra confirma.
    - 'sem_confirmacao': só a planilha administrativa registra o vínculo.
    """
    professores_aluno = professores_declarados_pelo_aluno.get((id_lattes_aluno, nivel), set())
    coorientadores = coorientadores_por_aluno_nivel.get((id_lattes_aluno, nivel), set())
    divergente = bool(professores_aluno) and not (professores_aluno & coorientadores)

    if divergente:
        return 'divergente'
    if encontrado_no_lattes and confirmado_pelo_aluno:
        return 'confirmado'
    if encontrado_no_lattes or confirmado_pelo_aluno:
        return 'confirmado_parcial'
    return 'sem_confirmacao'


df_situacao_resolvido = df_situacao_bruto[df_situacao_bruto['id_lattes_professor'].notna()].copy()
df_situacao_resolvido['encontrado_no_lattes'] = df_situacao_resolvido.apply(
    lambda linha: aluno_encontrado_no_lattes(
        linha['id_lattes_aluno'], linha['id_lattes_professor'], linha['M/D']
    ),
    axis=1,
)

_formacao_aluno_aplicada = df_situacao_resolvido.apply(
    lambda linha: dados_formacao_aluno(linha['id_lattes_aluno'], linha['M/D'], linha['id_lattes_professor']),
    axis=1,
    result_type='expand',
)
_formacao_aluno_aplicada.columns = ['titulo_trabalho_aluno', 'ano_obtencao_aluno', 'confirmado_pelo_aluno']
df_situacao_resolvido = pd.concat([df_situacao_resolvido, _formacao_aluno_aplicada], axis=1)

df_situacao_resolvido['nivel_confianca'] = df_situacao_resolvido.apply(
    lambda linha: nivel_de_confianca(
        linha['id_lattes_aluno'], linha['M/D'], linha['encontrado_no_lattes'], linha['confirmado_pelo_aluno']
    ),
    axis=1,
)

df_situacao_orientandos = (
    df_situacao_resolvido
    .rename(columns={
        'Aluno': 'nome_aluno', 'M/D': 'nivel', 'Ano': 'ano_ingresso',
        'orientador_texto': 'orientador_texto_original',
    })
    [[
        'id_lattes_professor', 'id_lattes_aluno', 'nome_aluno', 'nivel',
        'ano_ingresso', 'orientador_texto_original', 'encontrado_no_lattes',
        'titulo_trabalho_aluno', 'ano_obtencao_aluno', 'confirmado_pelo_aluno',
        'nivel_confianca',
    ]]
    .reset_index(drop=True)
)

print(f"Total de vínculos aluno-orientador avaliados: {len(df_situacao_orientandos)}")
print(f"Encontrados no Lattes do orientador:   {int(df_situacao_orientandos['encontrado_no_lattes'].sum())}")
print(f"Faltando no Lattes do orientador:      {int((~df_situacao_orientandos['encontrado_no_lattes']).sum())}")
print("\nDistribuição por nível de confiança:")
print(df_situacao_orientandos['nivel_confianca'].value_counts())
display(df_situacao_orientandos.head(10))

In [39]:
print("Persistindo 'tb_situacao_orientandos' no DuckDB (conexão própria, independente da Seção 13)...")
con_situacao = duckdb.connect(ARQUIVO_DUCKDB_DESTINO)

con_situacao.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_situacao_orientando;")
con_situacao.execute("""
    CREATE TABLE IF NOT EXISTS tb_situacao_orientandos (
        id_situacao INTEGER PRIMARY KEY DEFAULT nextval('seq_id_situacao_orientando'),
        id_lattes_professor VARCHAR,
        id_lattes_aluno VARCHAR,
        nome_aluno VARCHAR NOT NULL,
        nivel VARCHAR,
        ano_ingresso INTEGER,
        orientador_texto_original VARCHAR,
        encontrado_no_lattes BOOLEAN,
        titulo_trabalho_aluno VARCHAR,
        ano_obtencao_aluno INTEGER,
        confirmado_pelo_aluno BOOLEAN,
        nivel_confianca VARCHAR,
        FOREIGN KEY (id_lattes_professor) REFERENCES tb_professores(id_lattes)
    );
""")

# Migração aditiva para bancos já existentes com o schema antigo (sem as
# quatro colunas novas da triangulação com a Seção 11.2.2) -- não apaga nem
# recria nada, só adiciona o que faltar.
for coluna, tipo in [
    ('titulo_trabalho_aluno', 'VARCHAR'),
    ('ano_obtencao_aluno', 'INTEGER'),
    ('confirmado_pelo_aluno', 'BOOLEAN'),
    ('nivel_confianca', 'VARCHAR'),
]:
    con_situacao.execute(f"ALTER TABLE tb_situacao_orientandos ADD COLUMN IF NOT EXISTS {coluna} {tipo};")

con_situacao.execute("DELETE FROM tb_situacao_orientandos")

if not df_situacao_orientandos.empty:
    con_situacao.execute("""
        INSERT INTO tb_situacao_orientandos (
            id_lattes_professor, id_lattes_aluno, nome_aluno, nivel,
            ano_ingresso, orientador_texto_original, encontrado_no_lattes,
            titulo_trabalho_aluno, ano_obtencao_aluno, confirmado_pelo_aluno,
            nivel_confianca
        )
        SELECT
            id_lattes_professor, id_lattes_aluno, nome_aluno, nivel,
            ano_ingresso, orientador_texto_original, encontrado_no_lattes,
            titulo_trabalho_aluno, ano_obtencao_aluno, confirmado_pelo_aluno,
            nivel_confianca
        FROM df_situacao_orientandos
    """)

con_situacao.close()
print(
    f"Tabela 'tb_situacao_orientandos' atualizada em '{ARQUIVO_DUCKDB_DESTINO}' "
    f"({len(df_situacao_orientandos)} vínculo(s) aluno-orientador)."
)

Persistindo 'tb_situacao_orientandos' no DuckDB (conexão própria, independente da Seção 13)...
Tabela 'tb_situacao_orientandos' atualizada em 'pesquisadores_teste.duckdb' (328 vínculo(s) aluno-orientador).


## 15. Anos de Credenciamento dos Docentes

`tb_professores.data_ingresso` responde "a partir de quando a produção deste
docente conta", o que só descreve corretamente quem entrou no programa e nunca
mais saiu. Esta seção acrescenta a informação mais fina que o credenciamento
realmente exige: **quais anos, um a um, cada docente esteve credenciado** — o
que dá conta de descredenciamento, recredenciamento posterior e vigências com
lacunas no meio.

Fonte: `dados_brutos/credenciamento_professores.csv`, com uma linha por docente
e os anos separados por `;`:

```
id_lattes,nome_referencia,anos_credenciamento
1420784392366957,Marta Lima de Queirós Mattoso,2013;2014;2015;2016
```

> **Atenção:** enquanto o registro administrativo oficial do programa não
> existir, esse CSV é gerado com anos **ALEATÓRIOS** por
> `gerar_credenciamento_aleatorio.py` (semente fixa, então os números não mudam
> sozinhos entre execuções). Nada que sai da vigência serve como número oficial
> até o arquivo ser substituído pelo registro real. O `nome_referencia` está lá
> só para leitura humana na hora de editar a planilha: **o casamento é sempre
> pelo `id_lattes`**, exato, nunca por nome.

O resultado é `tb_credenciamento_anos(id_lattes, ano)` — uma linha por par
(docente, ano). O formato longo é o que permite ao `app.py` recortar a produção
com um `EXISTS` direto contra o ano da publicação, sem precisar interpretar
intervalos.

A leitura do CSV e a escrita da tabela ficam em `credenciamento.py`, a regra
única — as células abaixo só a chamam. O mesmo módulo tem uma CLI
(`python credenciamento.py --db pesquisadores_teste.duckdb`) que reaplica a
tabela a um banco já pronto quando só o CSV mudou, sem repetir o pipeline
inteiro.

Seção inteiramente aditiva, no mesmo molde da Seção 14: não lê nem modifica
nenhuma variável usada pela Seção 13, e persiste numa tabela nova com sua
própria conexão DuckDB. Se o CSV não existir, a tabela é criada vazia e o
`app.py` simplesmente não oferece o regime de vigência.


In [ ]:
print("Carregando os anos de credenciamento dos docentes...")

# A leitura/validação do CSV vive em `credenciamento.py`, e não aqui, para não
# divergir da CLI que aplica a mesma tabela a um banco já pronto
# (`python credenciamento.py --db ...`) -- mesmo arranjo de `dedup_publicacoes`.
import credenciamento as cred

# Conjunto de docentes válidos: a tabela é filha de tb_professores (FOREIGN
# KEY), então um id_lattes do CSV fora do cadastro não pode entrar.
ids_professores = set(df_pessoas['id_lattes'].astype(str)) if not df_pessoas.empty else set()

if Path(CAMINHO_CSV_CREDENCIAMENTO).exists():
    df_credenciamento_anos, ids_desconhecidos, anos_invalidos = cred.ler_csv_credenciamento(
        CAMINHO_CSV_CREDENCIAMENTO, ids_professores
    )
    for aviso in cred.formatar_avisos(ids_desconhecidos, anos_invalidos, CAMINHO_CSV_CREDENCIAMENTO):
        print(aviso)
else:
    print(
        f"AVISO: '{CAMINHO_CSV_CREDENCIAMENTO}' não encontrado -- 'tb_credenciamento_anos' "
        "ficará vazia e o regime de vigência não aparecerá no app. "
        "Rode 'python gerar_credenciamento_aleatorio.py' para gerar uma versão provisória."
    )
    df_credenciamento_anos = pd.DataFrame(columns=['id_lattes', 'ano'])

docentes_com_vigencia = (
    df_credenciamento_anos['id_lattes'].nunique() if not df_credenciamento_anos.empty else 0
)
print(
    f"Anos de credenciamento carregados: {len(df_credenciamento_anos)} par(es) (docente, ano) "
    f"para {docentes_com_vigencia} de {len(ids_professores)} docente(s) cadastrado(s)."
)


In [ ]:
print("Persistindo 'tb_credenciamento_anos' no DuckDB (conexão própria, independente da Seção 13)...")
con_credenciamento = duckdb.connect(ARQUIVO_DUCKDB_DESTINO)

cred.persistir_credenciamento(con_credenciamento, df_credenciamento_anos)
resumo_credenciamento = con_credenciamento.execute(cred.SQL_RESUMO).df()

con_credenciamento.close()
print(f"Tabela 'tb_credenciamento_anos' atualizada em '{ARQUIVO_DUCKDB_DESTINO}'.")
display(resumo_credenciamento)
